# Description

## Business Problem

This project was created to solve a real business problem. The company had a table of customers whose addresses were in string field where the users could write the whole addresses themselves. After some changes in business needs, the company had to split this field into proper columns: suburb, postcode, state, street name, street number, etc.

Since the company had more than 10k customers in the table, the script was created to help with this task. Also, the system used by the company would continue to be the same, that is, just one big string field for the address, so the script would still have to be used to continue to split the addresses into the columns.

To be able to complete the task, the fields had to be split, there was a separate address table to store the addresses and generate an ID that would also be used. To store the addresses in this table, I had to use other dimension tables to get the ID of the suburb and postcode. 

The expected address format in the string was:

```
                {Address1}
                {Address2}
{StreetNumber} {StreetName} {StreetType}
       {Suburb} {State} {Postcode}
```

One example of this would be:

```
{Example Civil Pty Ltd & Sample Builders Pty Ltd}                         ---> Address1
                    {T/As Example Road JV}                                ---> Address2
            {Level 2, 120-124} {Sample}   {Street}                        ---> StreetNo, Name and Type
                    {Abbotsford} {VIC} {3067}                             ---> Suburb, State, Postcode
```
We also had the PO Boxes and Privete Bags addresses:

```
  PO Box 1234
Rosebery NSW 1445
```

The only difference is that the whole first line of PO Boxes was intended to be in a specific column.

If the address was not in this format, the script would not work for it, and it should be updated.

All customer data in this repository is fictional, generated for demonstration only.

The sript was able to update more than 90% of the cases.

## Problems found while developing the project

### Problems that could be fixed creating functions

As the field was a big whole field, where the users could write the addresses as a whole, it was very easy to make mistakes and typos.

Here are some examples of challenging existing cases:

- Name of the suburb right but with the wrong postcode:

```
   PO Box 4321
Shepparton VIC 3631  ---> correct postcode would be 3630
```

- Name of the suburb wrong but with the correct postcode:

```
   PO Box 4321
Sheppaton VIC 3630  ---> correct suburb would be Shepparton
```

This problem happened because to split the fields, I needed to find the ID of the suburb and postcode in a table, so if they did not match, I had a problem.

- Order of the suburb name is different:

```
        PO Box W100
WESTFIELD PARRAMATTA NSW 2150   ---> it should be PARRAMATTA WESTFIELD
```

- Number of words of the suburb did not match the number of words the system had for that suburb and postcode:

```
    12 Example Road
WESTFIELD PENRITH NSW 2750     ---> it should be PENRITH
```

To fix these problems, 3 functions were created:

1. find_matching_suburb:

If there was no exact match with postcode and suburb in the system's list, the code uses this function to try to find a suburb with a similar name. It uses the fuzzywuzzy library, which allows me to find a percentage of match between 2 words. I set that the words had to have at least 80% of a match for the suburb to be chosen.

For this one, the number of words in the suburb had to be the same. For example, WESTFIELD PENRITH would not match with PENRITH.

2. find_matching_suburb_split

If the first function did not work, the code tries to find the same percentage of match, but now taking into account that the suburb could have more or less words than what was written by the user. So it tries to find the best match for postcode and suburb.

3. find_matching_suburb_split_state

The last step if the first 2 ones do not work is to also use the state to try to find the best match. Now the code uses both suburb and state to try to find the best match.

If none of these work, the code generates an error and adds the customer to the skip list, and moves on to the next one.

### Problems that caused the script to skip the address

Here are some examples of problems that the code could not fix, so when they happened, the code generates an error, add the error and the customer id to the error dataset.

- No postcodes

```
Level 5
2/100 Sample Street
 Sydney NSW
```

- Structure of the address incorrect:
```
Level 10
1 Example Street
Sydney NSW 2000 Australia
```

- No state found

```
17 Sample Road
Caringbah South 2229
```

- No street number

```
Example Street
Kurrajong NSW 2754
```

- No street name

```
Bldg 10, Room 200
Darlington  NSW  2006
```

These are just some cases, we had much more, but it's enough to ilustrate the problems.

# Importing the libraries

In [ ]:
import pandas as pd
import pyodbc
import re
from fuzzywuzzy import fuzz
from datetime import datetime
import os
import time
import xlsxwriter
import requests

# Creating the functions

In [ ]:
# Defining the function the will be used to find the the index of the last number of the string

# The street name starts after the last number

def find_last_number_index(input_string):
    # Find all numbers and single letters right after numbers in the string
    matches = re.finditer(r'(\d+[A-Za-z]?)(?![A-Za-z])', input_string)
    
    # If there are no matches, return -1
    if not matches:
        return -1

    # Find the last match
    last_match = None
    for match in matches:
        last_match = match
    
    if last_match:
        last_value = last_match.group()
        last_value_index = input_string.rfind(last_value)
        return last_value_index + len(last_value) - 1
    else:
        return -1

In [ ]:
# Function to find a suburb match based on fuzzy matching
def find_matching_suburb(suburb, postcode, suburbs):
    best_match = None
    best_match_score = 0
    
    # Tokenize and sort the words in the input suburb
    input_suburb_words = sorted(suburb.split())

    for index, row in suburbs.iterrows():
        current_suburb = row['SuburbName']
        current_postcode = row['Postcode']
        
        # Tokenize and sort the words in the current suburb
        current_suburb_words = sorted(current_suburb.upper().split())

        suburb_match_score = fuzz.partial_ratio(input_suburb_words, current_suburb_words)
        
        # You can adjust this threshold as needed to consider a match
        if suburb_match_score > best_match_score and suburb_match_score >= 80 and postcode == current_postcode and len(current_suburb.split()) == len(suburb.split()):
            best_match = current_suburb
            best_match_score = suburb_match_score

    return best_match

In [ ]:
# Function to find a suburb match based on fuzzy matching
def find_matching_suburb_split(suburb, postcode, suburbs):
    best_match = None
    best_match_score = 0

    for index, row in suburbs.iterrows():
        current_suburb = row['SuburbName']
        current_postcode = row['Postcode']

        suburb_match_score = fuzz.partial_ratio(suburb, current_suburb.upper())
        
        # You can adjust this threshold as needed to consider a match
        if suburb_match_score > best_match_score and suburb_match_score >= 80 and postcode == current_postcode:
            best_match = current_suburb
            best_match_score = suburb_match_score

    return best_match

In [ ]:
# Function to find a suburb match based on fuzzy matching
def find_matching_suburb_split_state(suburb, state, postcode, suburbs):
    best_match = None
    best_post_code = None
    best_match_score = 0
    best_postcode_diff = float('inf')  # Initialize with positive infinity

    for index, row in suburbs.iterrows():
        current_suburb = row['SuburbName']
        current_state = row['StateName']
        current_postcode = row['Postcode']

        # Calculate the similarity score between the input suburb and the current suburb
        suburb_match_score = fuzz.partial_ratio(suburb, current_suburb.upper())
                
        # You can adjust this threshold as needed to consider a match
        if (
            suburb_match_score >= best_match_score
            and suburb_match_score >= 80
            and current_state == state
        ):
            # Calculate the absolute difference between the current postcode and the target postcode
            postcode_diff = abs(int(current_postcode) - int(postcode))
            
            # Check if the current suburb has a postcode closer to the target postcode
            if postcode_diff < best_postcode_diff:
                best_match = current_suburb
                best_match_score = suburb_match_score
                best_postcode_diff = postcode_diff
                best_post_code = current_postcode

    return best_match, best_post_code

# Creating the objects that will be used in the code

## Connection settings

In [ ]:
server = 'sampleserver'
database = 'sample_database'
username = 'sampleUser'
password = 'SamplePassword'

## Columns for the datasets

In [ ]:
columns = ['Customer_Code', 'ID', 'Billing_ADR_Address_ID', 'Billing_Address', 'Billing_StreetNo', 'Billing_StreetName', 'Billing_StreetType',
           'Billing_Suburb', 'Billing_State', 'Billing_Postcode', 'Billing_Customisable_Value', 'Billing_Address2', 
           
           'Delivery_ADR_Address_ID', 'Delivery_Address', 'Delivery_StreetNo','Delivery_StreetName', 'Delivery_StreetType', 'Delivery_Suburb',
           'Delivery_State', 'Delivery_Postcode','Delivery_Customisable_Value' ,'Delivery_Address2']

columns

## Error Report Dataset

In [ ]:
billing_error_columns = ['Customer_Code', 'ID', 'Billing_Address', 'Billing Error Type']

delivey_error_columns = ['Customer_Code', 'ID', 'Delivery_Address', 'Delivery Error Type']

Billing_Error_Report = pd.DataFrame(columns=billing_error_columns)

Delivery_Error_Report = pd.DataFrame(columns=delivey_error_columns)

Billing_Error_Report

## Start time and Last Modified Date in the TBL_DBT_Address_Updates_Log table

In [ ]:
start_date = datetime.now()

To assist with the code, a table was created that would insert the information about the script runs. Every time the script runs, at the end, it inserts into a table information such as start datetime, end datetime, last modified datetime, number of rows affected, etc.

This part would make a connection with SQL Server to get the information about the last time the customer table was modified when the script ran for the last time. So it would make a query to the customer table with the modifications after that time.

I changed the connection to a file connection for studies proposes.

In [ ]:
# Establish the connection with Windows authentication

#conn = pyodbc.connect(
#    f'DRIVER=ODBC Driver 17 for SQL Server;SERVER={server};DATABASE={database};UID={username};PWD={password};'
#)

# Create a cursor object to interact with the database
#cursor = conn.cursor()

select_last_date = """
SELECT MAX([Last_Modified_Date]) AS [Last_Modified_Date]
  FROM [sample_database].[dbo].[TBL_DBT_Address_Updates_Log]
"""

select_last_date = pd.read_excel("TBL_DBT_Address_Updates_Log.xlsx")

# Convert 'Last_Modified_Date' column to datetime format
select_last_date['Last_Modified_Date'] = pd.to_datetime(select_last_date['Last_Modified_Date'])

# Get the maximum date
select_last_date = select_last_date['Last_Modified_Date'].max()

#cursor.execute(select_last_date)
#last_finish_date = cursor.fetchone()[0]

#cursor.close()
#conn.close()

last_finish_date = select_last_date

last_finish_date = last_finish_date.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]

last_finish_date

## Last Modified Date

This part basically gets the last modified date from the audit table, which will be used to make some comparisons at the end.

In [ ]:
# Establish the connection with Windows authentication
conn = pyodbc.connect(
    f'DRIVER=ODBC Driver 17 for SQL Server;SERVER={server};DATABASE={database};UID={username};PWD={password};'
)

# Create a cursor object to interact with the database
cursor = conn.cursor()


select_modified_dates = f"""
SELECT MAX([Audit_DateTime]) AS Max_Audit_DateTime
FROM [sample_database].[dbo].[VA_DBT_Master]
WHERE [Audited_Field] IN ('Billing_Address', 'Delivery_Address') 
AND Operation IN ('U', 'I') 
AND Audit_DateTime > '{last_finish_date}'
AND Audit_User NOT LIKE 'ServiceAccount'

"""

cursor.execute(select_modified_dates)
last_modified_date = cursor.fetchone()[0]

cursor.close()
conn.close()

if last_modified_date is None:
    last_modified_date = last_finish_date
else:
    last_modified_date = last_modified_date.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]

last_modified_date

# Separating the addresses

## Billing Address

### First Group: the standard data

This part would make a connection with the database to get the Billing Address of the customers. Some parts were replaced for studies purposes.

The original data was modified to preserve sensitive information.

In [ ]:
# Establish the connection with Windows authentication
#conn = pyodbc.connect(
#    f'DRIVER=ODBC Driver 17 for SQL Server;SERVER={server};DATABASE={database};UID={username};PWD={password};'
#)

# Create a cursor object to interact with the database
#cursor = conn.cursor()


select_billing_first = f"""

WITH UpdateSelection AS (

SELECT DISTINCT [Source_Id],[Audit_DateTime]
FROM [sample_database].[dbo].[VA_DBT_Master]
WHERE [Audited_Field] IN ('Billing_Address') 
AND Operation IN ('U', 'I') 
AND Audit_DateTime > '{last_finish_date}'
AND Audit_User NOT LIKE 'ServiceAccount'

AND [After_Value] IS NOT NULL AND [After_Value] NOT LIKE 'TBA%' AND NOT

([After_Value]  LIKE '%Locked Bag%' OR [After_Value] LIKE '%PO BOX%' OR [After_Value] LIKE '%P.O. BOX%' OR [After_Value] LIKE '%P.O BOX%'
 OR [After_Value] LIKE '%P O Box%' OR [After_Value] LIKE '%Private Bag%')

)

SELECT     
    [Customer_Code],
    [ID],
    REPLACE([Billing_Address],CHAR(13)+CHAR(10),'_') AS [Billing_Address]
FROM [sample_database].[dbo].[DBT_Master]

WHERE 

[Billing_Address] IS NOT NULL AND [Billing_Address] NOT LIKE 'TBA%' AND NOT

([Billing_Address]  LIKE '%Locked Bag%' OR [Billing_Address] LIKE '%PO BOX%' OR [Billing_Address] LIKE '%P.O. BOX%' OR [Billing_Address] LIKE '%P.O BOX%'
 OR [Billing_Address] LIKE '%P O Box%' OR [Billing_Address] LIKE '%Private Bag%')

AND [ID] IN (SELECT DISTINCT [Source_Id]
            FROM UpdateSelection)
            
        """


#df = pd.read_sql_query(select_billing_first, conn)

#cursor.close()
#conn.close()

df = pd.read_excel("Billing Address.xlsx")

df

In [ ]:
billing_first_group = pd.DataFrame(columns=columns)

billing_first_group['Customer_Code'] = df['Customer_Code']
billing_first_group['ID'] = df['ID']
billing_first_group['Billing_Address'] = df['Billing_Address']

billing_first_group

#### Post Code, State and Suburb

In [ ]:
# testing if there's any row not in the pattern

Billing_IDs_To_Exclude = []

error_reason = ''

states = ['NSW', 'VIC', 'QLD', 'SA', 'WA', 'TAS', 'ACT', 'NT']

for index, row in df.iterrows():
    
    customer_ID = row['ID']
    
    parts = row['Billing_Address'].strip('_').strip('.').split('_') # spliting the string by _ that represents a line break
    
    
    #tests if the lenght of the Billing address is less than 2 lines, if yes, the address is not complete and it skips the rest of the code
    
    if len(parts) < 2:
        Billing_IDs_To_Exclude.append(customer_ID)
        error_reason += 'Address is not complete. It should have at least 2 lines!\n'
        
        new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Billing_Address': row['Billing_Address'].replace("_", "\n"),
                                                    'Billing Error Type': error_reason
                        }])
        
        Billing_Error_Report = pd.concat([Billing_Error_Report, new_line],ignore_index=True)
        
        error_reason = ''
        
        continue
        
    # if no, it tests if the last line, which should be the suburb details, have at least 3 elements, if yes, it gets the state and the code continues
    # case it's less than 3 elements, it skips the rest of the code 
    
    else:

        suburb_details = parts[-1].split() # the last part is usually where the Suburb, post code and state are
        
        if len(suburb_details) > 2:
            state = suburb_details[-2].upper().strip(',').strip('`').strip('.')
        else:
            # Handle the case when the length is less than 3
            state = ''
            error_reason += 'Wrong Format. Please make sure the format looks like:\n{Address1}\n{Address2}\n{StreetNumber} {StreetName} {StreetType}\n{Suburb} {State} {Postcode}\n'
            Billing_IDs_To_Exclude.append(customer_ID)
            
            new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Billing_Address': row['Billing_Address'].replace("_", "\n"),
                                                    'Billing Error Type': error_reason
                        }])
            
            Billing_Error_Report = pd.concat([Billing_Error_Report, new_line],ignore_index=True)
        
            error_reason = ''
        
            continue
            
           #----------------------------------------------------------------------------------------------------------------------------------------------------------# 
            

        post_code = suburb_details[-1].strip(',').strip('`').strip().strip('.') # the post code is normally the last word in the last line

        suburb = ' '.join(suburb_details[:-2]).upper().strip(',').strip('.') # suburb is the rest, it can be 2 or more words, so we join them with a space

        # Check if the post_code looks like 4 numbers
        is_post_code_valid = re.match(r'^\d{4}$', post_code) is not None

        # Check if the state looks like 3 or 2 letters
        is_state_valid = re.match(r'^[A-Za-z]{2,3}$', state) is not None
            
        
        if is_post_code_valid == False:
            
            if isinstance(post_code, int):
                error_reason += 'Postcode invalid. It should be 4 numbers.\n'
                Billing_IDs_To_Exclude.append(customer_ID)
                
                
            if not any(re.match(r'^\d{4}$', str(item)) is not None for item in suburb_details):
                error_reason += 'Postcode Missing.\n'
                Billing_IDs_To_Exclude.append(customer_ID)
                
                if not any(state in suburb_details for state in states):
                    error_reason += 'State Missing.\n'
                    
                    
                new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                        'ID': customer_ID,
                                                        'Billing_Address': row['Billing_Address'].replace("_", "\n"),
                                                        'Billing Error Type': error_reason
                            }])

                Billing_Error_Report = pd.concat([Billing_Error_Report, new_line],ignore_index=True)

                error_reason = ''
                
                continue

                
            else:
                
                if not any(state in suburb_details for state in states):
                    error_reason += 'State Missing.\n'
                    Billing_IDs_To_Exclude.append(customer_ID)
                    
                    new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Billing_Address': row['Billing_Address'].replace("_", "\n"),
                                                    'Billing Error Type': error_reason
                        }])
            
                    Billing_Error_Report = pd.concat([Billing_Error_Report, new_line],ignore_index=True)
                
                    error_reason = ''
                    
                    continue
                else:
                    error_reason += 'Wrong Format. Please make sure the format looks like:\n{Address1}\n{Address2}\n{StreetNumber} {StreetName} {StreetType}\n{Suburb} {State} {Postcode}\n'
                    Billing_IDs_To_Exclude.append(customer_ID)
                    
                    new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Billing_Address': row['Billing_Address'].replace("_", "\n"),
                                                    'Billing Error Type': error_reason
                        }])
            
                    Billing_Error_Report = pd.concat([Billing_Error_Report, new_line],ignore_index=True)
                
                    error_reason = ''
                    
                    continue
                    
                    
        
        if is_state_valid == False:
            
            if not any(state in suburb_details for state in states):
                error_reason += 'State Missing.\n'
                Billing_IDs_To_Exclude.append(customer_ID)
            else:
                Billing_IDs_To_Exclude.append(customer_ID)
                error_reason += 'State invalid. It should be 2 or 3 characters.\n'
                
                
        if suburb == '':
            error_reason += 'Suburb is Missing.'
            Billing_IDs_To_Exclude.append(customer_ID)

            
        if is_post_code_valid is False or is_state_valid is False or suburb == '':
            new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Billing_Address': row['Billing_Address'].replace("_", "\n"),
                                                    'Billing Error Type': error_reason
                        }])
            
            Billing_Error_Report = pd.concat([Billing_Error_Report, new_line],ignore_index=True)
                
        error_reason = ''

In [ ]:
Billing_Error_Report

In [ ]:
# Firt, we separate the Suburb, Post Code and State and put them into the new dataset

# The pattern we are looking for is: Condell Park NSW 2200 (Suburb State Postcode)

for index, row in df.iterrows():
    
    customer_ID = row['ID']
    
    if customer_ID in Billing_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Billing_Address'].strip('_').strip('.').split('_') # spliting the string by _ that represents a line break

        suburb_details = parts[-1].split() # the last part is usually where the Suburb, post code and state are

        post_code = suburb_details[-1].strip(',').strip('`').strip().strip('.') # the post code is normally the last word in the last line

        state = suburb_details[-2].upper().strip(',').strip('`').strip('.') # the state is normally the second to last

        suburb = ' '.join(suburb_details[:-2]).upper().strip(',').strip('.') # suburb is the rest, it can be 2 or more words, so we join them with a space

        billing_address = '\n'.join(parts[:-1])

        billing_address += f'\n{suburb} {state} {post_code}'

        billing_first_group.loc[billing_first_group['ID'] == customer_ID, 'Billing_Postcode'] = post_code
        billing_first_group.loc[billing_first_group['ID'] == customer_ID, 'Billing_State'] = state
        billing_first_group.loc[billing_first_group['ID'] == customer_ID, 'Billing_Suburb'] = suburb
        billing_first_group.loc[billing_first_group['ID'] == customer_ID, 'Billing_Address'] = billing_address


#### Street Number and Street Name

In [ ]:
# testing if there's any row not in the pattern

error_reason = ''

for index, row in df.iterrows():
    
    customer_ID = row['ID']
    
    if customer_ID in Billing_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Billing_Address'].strip('_').strip('.').split('_')

        street_details = parts[-2] # geting the second group of the split

        last_index = find_last_number_index(street_details) # geting the index of the last number, which represents the street number

        street_no = street_details[:last_index+1].strip(',').strip('.') # street no is everything until the index of the last number

        street_name = street_details[last_index+1:] # street name starts right after the last number

        street_name = street_name.strip(',').strip('`').strip().strip('.').title()
        
        if street_no == '':
            
            if not ("Cnr" in street_details or "Corner" in street_details):
                Billing_IDs_To_Exclude.append(customer_ID)
                error_reason += 'No street number found!\n'
            else:
                street_no = 'NULL'
            
        if street_name == '':
            Billing_IDs_To_Exclude.append(customer_ID)
            error_reason += 'No street name found!.\n'
            
            
        if street_no == '' or street_name == '':
            new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Billing_Address': row['Billing_Address'].replace("_", "\n"),
                                                    'Billing Error Type': error_reason
                        }])
            
            Billing_Error_Report = pd.concat([Billing_Error_Report, new_line],ignore_index=True)
                
        error_reason = ''

In [ ]:
# Second, we get the street detail, that is normally the second group of the split

for index, row in df.iterrows():
    
    customer_ID = row['ID']
    
    if customer_ID in Billing_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Billing_Address'].strip('_').strip('.').split('_')

        street_details = parts[-2] # geting the second group of the split

        last_index = find_last_number_index(street_details) # geting the index of the last number, which represents the street number

        street_no = street_details[:last_index+1].strip(',').strip('.') # street no is everything until the index of the last number

        street_name = street_details[last_index+1:] # street name starts right after the last number

        street_name = street_name.strip(',').strip('`').strip().strip('.').strip(',').title()

        billing_first_group.loc[billing_first_group['ID'] == customer_ID, 'Billing_StreetNo'] = street_no
        billing_first_group.loc[billing_first_group['ID'] == customer_ID, 'Billing_StreetName'] = street_name

#### Billing Address 2

In [ ]:
# Third, we get what we have in the third group for those who have something and put in the Billing Address 2 column
for index, row in df.iterrows():
    
    customer_ID = row['ID']
    
    if customer_ID in Billing_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Billing_Address'].strip('_').strip('.').split('_')

        trading_as = []
        adress2 = []

        if len(parts) > 2:

            for part in parts[:-2]:
                if part.lower().startswith("t/a") or part.lower().startswith("t/as"):
                    trading_as.append(part.strip().strip(',').strip('.'))
                else:
                    adress2.append(part.strip().strip(',').strip('.'))

            if len(adress2) > 0:
                billing_address2 = '\n'.join(adress2) # geting the second group of the split
                billing_address2 = billing_address2.strip('\n').strip('.')
                billing_first_group.loc[billing_first_group['ID'] == customer_ID, 'Billing_Address2'] = billing_address2

            if len(trading_as) > 0:
                billing_ta = '\n'.join(trading_as)
                billing_ta = billing_ta.strip().strip(',').strip('.')
                billing_first_group.loc[billing_first_group['ID'] == customer_ID, 'Billing_Customisable_Value'] = billing_ta
                billing_ta = None

#### Updating the Billing Address

In [ ]:
for index, row in df.iterrows():
    
    customer_ID = row['ID']

    if customer_ID in Billing_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Billing_Address'].strip('_').strip('.').split('_') # spliting the string by _ that represents a line break

        suburb_details = parts[-1].split() # the last part is usually where the Suburb, post code and state are 

        post_code = suburb_details[-1] # the post code is normally the last word in the last line

        state = suburb_details[-2].upper() # the state is normally the second to last

        suburb = ' '.join(suburb_details[:-2]).upper().strip(',').strip('.') # suburb is the rest, it can be 2 or more words, so we join them with a space

        billing_address = '\n'.join(parts[:-1])

        billing_address += f'\n{suburb} {state} {post_code}'

        billing_first_group.loc[billing_first_group['ID'] == customer_ID, 'Billing_Address'] = billing_address

In [ ]:
billing_first_group = billing_first_group[~billing_first_group['ID'].isin(Billing_IDs_To_Exclude)]

billing_first_group

### Second Group: PO Box and Locked Bag

In [ ]:
# Establish the connection with Windows authentication

#conn = pyodbc.connect(
#    f'DRIVER=ODBC Driver 17 for SQL Server;SERVER={server};DATABASE={database};UID={username};PWD={password};'
#)

# Create a cursor object to interact with the database

#cursor = conn.cursor()

select_billing_second = f"""

WITH UpdateSelection AS (

SELECT DISTINCT [Source_Id]
FROM [sample_database].[dbo].[VA_DBT_Master]
WHERE [Audited_Field] IN ('Billing_Address') 
AND Operation IN ('U', 'I')
AND Audit_DateTime > '{last_finish_date}'
AND Audit_User NOT LIKE 'ServiceAccount'

AND [After_Value] IS NOT NULL AND [After_Value] NOT LIKE 'TBA%' AND

([After_Value]  LIKE '%Locked Bag%' OR [After_Value] LIKE '%PO BOX%' OR [After_Value] LIKE '%P.O. BOX%' OR [After_Value] LIKE '%P.O BOX%'
 OR [After_Value] LIKE '%P O Box%' OR [After_Value] LIKE '%Private Bag%')

)

SELECT     
    [Customer_Code],
    [ID],
    REPLACE([Billing_Address],CHAR(13)+CHAR(10),'_') AS [Billing_Address]
FROM [sample_database].[dbo].[DBT_Master]

WHERE

[Billing_Address] IS NOT NULL AND [Billing_Address] NOT LIKE 'TBA%' AND

([Billing_Address]  LIKE '%Locked Bag%' OR [Billing_Address] LIKE '%PO BOX%' OR [Billing_Address] LIKE '%P.O. BOX%' OR [Billing_Address] LIKE '%P.O BOX%'
 OR [Billing_Address] LIKE '%P O Box%' OR [Billing_Address] LIKE '%Private Bag%')

AND [ID] IN (SELECT DISTINCT [Source_Id]
            FROM UpdateSelection)

        """
#df2 = pd.read_sql_query(select_billing_second, conn)

#cursor.close()
#conn.close()

df2 = pd.read_excel("Billing Address - PO Boxes.xlsx")

df2

In [ ]:
billing_second_group = pd.DataFrame(columns=columns)

billing_second_group['Customer_Code'] = df2['Customer_Code']
billing_second_group['ID'] = df2['ID']
billing_second_group['Billing_Address'] = df2['Billing_Address']

billing_second_group

#### Post Code, State and Suburb

In [ ]:
# testing if there's any row not in the pattern

error_reason = ''


for index, row in df2.iterrows():
    
    customer_ID = row['ID']
    
    ## attention with the ID 7112
    
    parts = row['Billing_Address'].strip('_').strip('.').split('_') # spliting the string by _ that represents a line break
    
    #tests if the lenght of the Billing address is less than 2 lines, if yes, the address is not complete and it skips the rest of the code
    
    if len(parts) < 2:
        Billing_IDs_To_Exclude.append(customer_ID)
        error_reason += 'Address is not complete. It should have at least 2 lines!\n'
        
        new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Billing_Address': row['Billing_Address'].replace("_", "\n"),
                                                    'Billing Error Type': error_reason
                        }])
        
        Billing_Error_Report = pd.concat([Billing_Error_Report, new_line],ignore_index=True)
        
        error_reason = ''
        
        continue
        
    # if no, it tests if the last line, which should be the suburb details, have at least 3 elements, if yes, it gets the state and the code continues
    # case it's less than 3 elements, it skips the rest of the code 
        
    else:
    
        suburb_details = parts[-1].split() # the last part is usually where the Suburb, post code and state are 
        
        if len(suburb_details) > 2:
            state = suburb_details[-2].upper().strip(',').strip('`').strip('.')
        else:
            # Handle the case when the length is less than 3
            state = ''
            error_reason += 'Wrong Format. Please make sure the format looks like:\n{Address1}\n{Address2}\n{StreetNumber} {StreetName} {StreetType}\n{Suburb} {State} {Postcode}\n'
            Billing_IDs_To_Exclude.append(customer_ID)
            
            new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Billing_Address': row['Billing_Address'].replace("_", "\n"),
                                                    'Billing Error Type': error_reason
                        }])
            
            Billing_Error_Report = pd.concat([Billing_Error_Report, new_line],ignore_index=True)
        
            error_reason = ''
        
            continue
            
           #----------------------------------------------------------------------------------------------------------------------------------------------------------# 
            

        post_code = suburb_details[-1].strip(',').strip('`').strip().strip('.') # the post code is normally the last word in the last line

        suburb = ' '.join(suburb_details[:-2]).upper().strip(',').strip('.') # suburb is the rest, it can be 2 or more words, so we join them with a space

        # Check if the post_code looks like 4 numbers
        is_post_code_valid = re.match(r'^\d{4}$', post_code) is not None

        # Check if the state looks like 3 or 2 letters
        is_state_valid = re.match(r'^[A-Za-z]{2,3}$', state) is not None
        
    
        keywords_to_exclude = ['Locked Bag', 'PO BOX', 'P.O. BOX', 'P.O BOX', 'P O Box', 'Private Bag']
            
        if any(keyword.upper() in suburb for keyword in keywords_to_exclude):
            Billing_IDs_To_Exclude.append(customer_ID)
            error_reason += 'Suburb contains address info.'
        
        if is_post_code_valid == False:
            
            Billing_IDs_To_Exclude.append(customer_ID)
            if isinstance(post_code, int):
                error_reason += 'Postcode invalid. It should be 4 numbers.\n'
                Billing_IDs_To_Exclude.append(customer_ID)
                
                
            if not any(re.match(r'^\d{4}$', str(item)) is not None for item in suburb_details):
                error_reason += 'Postcode Missing.\n'
                Billing_IDs_To_Exclude.append(customer_ID)
                
                if not any(state in suburb_details for state in states):
                    error_reason += 'State Missing.\n'
                    
                    
                new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                        'ID': customer_ID,
                                                        'Billing_Address': row['Billing_Address'].replace("_", "\n"),
                                                        'Billing Error Type': error_reason
                            }])

                Billing_Error_Report = pd.concat([Billing_Error_Report, new_line],ignore_index=True)

                error_reason = ''
                
                continue

                
            else:
                
                if not any(state in suburb_details for state in states):
                    error_reason += 'State Missing.\n'
                    Billing_IDs_To_Exclude.append(customer_ID)
                    
                    new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Billing_Address': row['Billing_Address'].replace("_", "\n"),
                                                    'Billing Error Type': error_reason
                        }])
            
                    Billing_Error_Report = pd.concat([Billing_Error_Report, new_line],ignore_index=True)
                
                    error_reason = ''
                    
                    continue
                else:
                    error_reason += 'Wrong Format. Please make sure the format looks like:\n{Address1}\n{Address2}\n{StreetNumber} {StreetName} {StreetType}\n{Suburb} {State} {Postcode}\n'
                    Billing_IDs_To_Exclude.append(customer_ID)
                    
                    new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Billing_Address': row['Billing_Address'].replace("_", "\n"),
                                                    'Billing Error Type': error_reason
                        }])
            
                    Billing_Error_Report = pd.concat([Billing_Error_Report, new_line],ignore_index=True)
                
                    error_reason = ''
                    
                    continue
                    
                    
        
        if is_state_valid == False:
            if not any(state in suburb_details for state in states):
                error_reason += 'State Missing.\n'
                Billing_IDs_To_Exclude.append(customer_ID)
            else:
                Billing_IDs_To_Exclude.append(customer_ID)
                error_reason += 'State invalid. It should be 2 or 3 characters.\n'
            
        
        if suburb == '':
            error_reason += 'Suburb is Missing.'
            Billing_IDs_To_Exclude.append(customer_ID)
        
            
        if is_post_code_valid is False or is_state_valid is False or suburb == '' or any(keyword.upper() in suburb for keyword in keywords_to_exclude):
            new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Billing_Address': row['Billing_Address'].replace("_", "\n"),
                                                    'Billing Error Type': error_reason
                        }])
            
            Billing_Error_Report = pd.concat([Billing_Error_Report, new_line],ignore_index=True)
                
        error_reason = ''

In [ ]:
# Firt, we separate the Suburb, Post Code and State and put them into the new dataset

# The pattern we are looking for is: Condell Park NSW 2200 (Suburb State Post Code)

for index, row in df2.iterrows():
    
    customer_ID = row['ID']
    
    if customer_ID in Billing_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Billing_Address'].strip('_').strip('.').split('_') # spliting the string by _ that represents a line break

        suburb_details = parts[-1].split() # the last part is usually where the Suburb, post code and state are 

        post_code = suburb_details[-1].strip(',').strip('`').strip().strip('.') # the post code is normally the last word in the last line

        state = suburb_details[-2].upper().strip(',').strip('`').strip('.') # the state is normally the second to last

        suburb = ' '.join(suburb_details[:-2]).upper().strip(',').strip('.') # suburb is the rest, it can be 2 or more words, so we join them with a space

        billing_second_group.loc[billing_second_group['ID'] == customer_ID, 'Billing_Postcode'] = post_code
        billing_second_group.loc[billing_second_group['ID'] == customer_ID, 'Billing_State'] = state
        billing_second_group.loc[billing_second_group['ID'] == customer_ID, 'Billing_Suburb'] = suburb

#### Billing Address 2

It was the billing address 2 the field that would receive the values, but after some problems in converting quotes in Salesforce, this field was changed to Billing_StreetName.

Reffer to this change in the TASK# 002370 - Split Billing Address - PO Box Cases (Baseplan)

In [ ]:
for index, row in df2.iterrows():
    
    customer_ID = row['ID']
    
    if customer_ID in Billing_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Billing_Address'].strip('_').strip('.').split('_') # spliting the string by _ that represents a line break

        trading_as = []
        adress2 = []
        streetname = []

        for part in parts[:-1]:
                if part.lower().startswith("t/a") or part.lower().startswith("t/as"):
                    trading_as.append(part.strip().strip(',').strip('.'))
                elif re.search(r'\bbox\b|\bbag\b', part, re.IGNORECASE):
                    streetname.append(part.strip().strip(',').strip('.'))
                else:                       
                    adress2.append(part.strip().strip(',').strip('.'))

        if len(adress2) > 0:
            billing_address2 = '\n'.join(adress2) # geting the second group of the split
            billing_address2 = billing_address2.strip('\n').strip().strip(',').strip('.')
            billing_second_group.loc[billing_second_group['ID'] == customer_ID, 'Billing_Address2'] = billing_address2
        
        if len(streetname) > 0:
            streetName = '\n'.join(streetname) # geting the second group of the split
            streetName.strip('\n').strip().strip(',').strip('.')            
            billing_second_group.loc[billing_second_group['ID'] == customer_ID, 'Billing_StreetName'] = streetname

        if len(trading_as) > 0:
            billing_ta = '\n'.join(trading_as)
            billing_ta = billing_ta.strip('\n').strip().strip(',').strip('.')
            billing_second_group.loc[billing_second_group['ID'] == customer_ID, 'Billing_Customisable_Value'] = billing_ta
            billing_ta = None

#### Updating the Billing Address

In [ ]:
for index, row in df2.iterrows():
    
    customer_ID = row['ID']
    
    if customer_ID in Billing_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Billing_Address'].strip('_').strip('.').split('_') # spliting the string by _ that represents a line break

        suburb_details = parts[-1].split() # the last part is usually where the Suburb, post code and state are

        post_code = suburb_details[-1].strip('.') # the post code is normally the last word in the last line

        state = suburb_details[-2].upper() # the state is normally the second to last

        suburb = ' '.join(suburb_details[:-2]).upper().strip(',').strip('.') # suburb is the rest, it can be 2 or more words, so we join them with a space

        billing_address = '\n'.join(parts[:-1])

        billing_address += f'\n{suburb} {state} {post_code}'

        billing_second_group.loc[billing_second_group['ID'] == customer_ID, 'Billing_Address'] = billing_address

In [ ]:
billing_second_group = billing_second_group[~billing_second_group['ID'].isin(Billing_IDs_To_Exclude)]

billing_second_group

## Delivery Address

### First Group: the standard data

In [ ]:
# Establish the connection with Windows authentication

#conn = pyodbc.connect(
#    f'DRIVER=ODBC Driver 17 for SQL Server;SERVER={server};DATABASE={database};UID={username};PWD={password};'
#)

# Create a cursor object to interact with the database
#cursor = conn.cursor()


select_delivery_first = f"""

WITH UpdateSelection AS (

SELECT DISTINCT [Source_Id]
FROM [sample_database].[dbo].[VA_DBT_Master]
WHERE [Audited_Field] IN ('Delivery_Address') 
AND Operation IN ('U', 'I') 
AND Audit_DateTime > '{last_finish_date}' 
AND Audit_User NOT LIKE 'ServiceAccount'

AND [After_Value] IS NOT NULL AND [After_Value] NOT LIKE 'TBA%' AND NOT

([After_Value]  LIKE '%Locked Bag%' OR [After_Value] LIKE '%PO BOX%' OR [After_Value] LIKE '%P.O. BOX%' OR [After_Value] LIKE '%P.O BOX%'
 OR [After_Value] LIKE '%P O Box%' OR [After_Value] LIKE '%Private Bag%')

)

SELECT     
    [Customer_Code],
    [ID],
    REPLACE([Delivery_Address],CHAR(13)+CHAR(10),'_') AS [Delivery_Address]
FROM [sample_database].[dbo].[DBT_Master]

WHERE 

[Delivery_Address] IS NOT NULL AND [Delivery_Address] NOT LIKE 'TBA%' AND NOT

([Delivery_Address]  LIKE '%Locked Bag%' OR [Delivery_Address] LIKE '%PO BOX%' OR [Delivery_Address] LIKE '%P.O. BOX%' OR [Delivery_Address] LIKE '%P.O BOX%'
 OR [Delivery_Address] LIKE '%P O Box%' OR [Delivery_Address] LIKE '%Private Bag%')

AND [ID] IN (SELECT DISTINCT [Source_Id]
            FROM UpdateSelection)
                    
"""

#df3 = pd.read_sql_query(select_delivery_first, conn)

#cursor.close()
#conn.close()

df3 = pd.read_excel("Delivery Address.xlsx")

df3

In [ ]:
delivery_first_group = pd.DataFrame(columns=columns)

delivery_first_group['Customer_Code'] = df3['Customer_Code']
delivery_first_group['ID'] = df3['ID']
delivery_first_group['Delivery_Address'] = df3['Delivery_Address']

delivery_first_group

#### Post Code, State and Suburb

In [ ]:
# testing if there's any row not in the pattern

Delivery_IDs_To_Exclude = []

error_reason = ''

for index, row in df3.iterrows():
    
    customer_ID = row['ID']
    
    parts = str(row['Delivery_Address']).strip('_').strip('.').split('_') # spliting the string by _ that represents a line break
    
    
    #tests if the lenght of the Billing address is less than 2 lines, if yes, the address is not complete and it skips the rest of the code

    if len(parts) < 2:
        Delivery_IDs_To_Exclude.append(customer_ID)
        error_reason += 'Address is not complete. It should have at least 2 lines!\n'
        
        new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Delivery_Address': row['Delivery_Address'].replace("_", "\n"),
                                                    'Delivery Error Type': error_reason
                        }])
        
        Delivery_Error_Report = pd.concat([Delivery_Error_Report, new_line],ignore_index=True)
        
        error_reason = ''
        
        continue
        
    else:
    
        suburb_details = parts[-1].split() # the last part is usually where the Suburb, post code and state are
        
        if len(suburb_details) >= 2:
            state = suburb_details[-2].upper().strip(',').strip('`').strip('.') # the state is normally the second to last
        else:
            # Handle the case when the length is less than 2
            state = ''
            error_reason += 'Wrong Format. Please make sure the format looks like:\n{Address1}\n{Address2}\n{StreetNumber} {StreetName} {StreetType}\n{Suburb} {State} {Postcode}\n'
            Delivery_IDs_To_Exclude.append(customer_ID)
            
            new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Delivery_Address': row['Delivery_Address'].replace("_", "\n"),
                                                    'Delivery Error Type': error_reason
                        }])
            
            Delivery_Error_Report = pd.concat([Delivery_Error_Report, new_line],ignore_index=True)
        
            error_reason = ''
        
            continue
            
           #----------------------------------------------------------------------------------------------------------------------------------------------------------# 

        post_code = suburb_details[-1].strip(',').strip('`').strip().strip('.').strip('.') # the post code is normally the last word in the last line

        suburb = ' '.join(suburb_details[:-2]).upper().strip(',').strip('.') # suburb is the rest, it can be 2 or more words, so we join them with a space

        # Check if the post_code looks like 4 numbers
        is_post_code_valid = re.match(r'^\d{4}$', post_code) is not None

        # Check if the state looks like 3 or 2 letters
        is_state_valid = re.match(r'^[A-Za-z]{2,3}$', state) is not None
        
            
        if is_post_code_valid == False:
            if isinstance(post_code, int):
                error_reason += 'Postcode invalid. It should be 4 numbers.\n'
                Delivery_IDs_To_Exclude.append(customer_ID)
                
                
            if not any(re.match(r'^\d{4}$', str(item)) is not None for item in suburb_details):
                error_reason += 'Postcode Missing.\n'
                Delivery_IDs_To_Exclude.append(customer_ID)
                
                if not any(state in suburb_details for state in states):
                    error_reason += 'State Missing.\n'
                    
                    
                new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Delivery_Address': row['Delivery_Address'].replace("_", "\n"),
                                                    'Delivery Error Type': error_reason
                        }])
            
                Delivery_Error_Report = pd.concat([Delivery_Error_Report, new_line],ignore_index=True)

                error_reason = ''
                
                continue

                
            else:
                
                if not any(state in suburb_details for state in states):
                    error_reason += 'State Missing.\n'
                    Delivery_IDs_To_Exclude.append(customer_ID)
                    
                    new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Delivery_Address': row['Delivery_Address'].replace("_", "\n"),
                                                    'Delivery Error Type': error_reason
                        }])
            
                    Delivery_Error_Report = pd.concat([Delivery_Error_Report, new_line],ignore_index=True)

                    error_reason = ''
                    
                    continue
                    
                else:
                    error_reason += 'Wrong Format. Please make sure the format looks like:\n{Address1}\n{Address2}\n{StreetNumber} {StreetName} {StreetType}\n{Suburb} {State} {Postcode}\n'
                    Delivery_IDs_To_Exclude.append(customer_ID)
                    
                    new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Delivery_Address': row['Delivery_Address'].replace("_", "\n"),
                                                    'Delivery Error Type': error_reason
                        }])
            
                    Delivery_Error_Report = pd.concat([Delivery_Error_Report, new_line],ignore_index=True)

                    error_reason = ''
                    
                    continue
                    
        
        if is_state_valid == False:
            
            if not any(state in suburb_details for state in states):
                error_reason += 'State Missing.\n'
                Delivery_IDs_To_Exclude.append(customer_ID)
            else:
                Delivery_IDs_To_Exclude.append(customer_ID)
                error_reason += 'State invalid. It should be 2 or 3 characters.\n'
                
        
        if suburb == '':
            error_reason += 'Suburb is Missing.'
            Delivery_IDs_To_Exclude.append(customer_ID)
            
            
        if is_post_code_valid is False or is_state_valid is False or suburb == '':
            new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Delivery_Address': row['Delivery_Address'].replace("_", "\n"),
                                                    'Delivery Error Type': error_reason
                        }])
            
            Delivery_Error_Report = pd.concat([Delivery_Error_Report, new_line],ignore_index=True)
                
        error_reason = ''

In [ ]:
# Firt, we separate the Suburb, Post Code and State and put them into the new dataset

# The pattern we are looking for is: Condell Park NSW 2200 (Suburb State Post Code)

for index, row in df3.iterrows():
    
    customer_ID = row['ID']
    
    if customer_ID in Delivery_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Delivery_Address'].strip('_').strip('.').split('_') # spliting the string by _ that represents a line break

        suburb_details = parts[-1].strip('.').split() # the last part is usually where the Suburb, post code and state are 

        post_code = suburb_details[-1].strip(',').strip('`').strip().strip('.') # the post code is normally the last word in the last line

        state = suburb_details[-2].upper().strip(',').strip('`').strip('.') # the state is normally the second to last

        suburb = ' '.join(suburb_details[:-2]).upper().strip(',').strip('.') # suburb is the rest, it can be 2 or more words, so we join them with a space

        delivery_first_group.loc[delivery_first_group['ID'] == customer_ID, 'Delivery_Postcode'] = post_code
        delivery_first_group.loc[delivery_first_group['ID'] == customer_ID, 'Delivery_State'] = state
        delivery_first_group.loc[delivery_first_group['ID'] == customer_ID, 'Delivery_Suburb'] = suburb

#### Street Number and Street Name

In [ ]:
# testing if the street number is empty, in case it if, it's not in the pattern

error_reason = ''

for index, row in df3.iterrows():
    
    customer_ID = row['ID']
    
    if customer_ID in Delivery_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Delivery_Address'].strip('_').strip('.').split('_')

        street_details = parts[-2] # geting the second group of the split

        last_index = find_last_number_index(street_details) # geting the index of the last number, which represents the street number

        street_no = street_details[:last_index+1].strip(',').strip('.') # street no is everything until the index of the last number

        street_name = street_details[last_index+1:] # street name starts right after the last number

        street_name = street_name.strip(',').strip('`').strip().strip('.').title()

        if street_no == '':
            
            if not ("Cnr" in street_details or "Corner" in street_details):
                Delivery_IDs_To_Exclude.append(customer_ID)
                error_reason += 'No street number found!\n'
            else:
                street_no = 'NULL'

        if street_name == '':
            Delivery_IDs_To_Exclude.append(customer_ID)
            error_reason += 'No street name found!\n'


        if street_no == '' or street_name == '':
            new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Delivery_Address': row['Delivery_Address'].replace("_", "\n"),
                                                    'Delivery Error Type': error_reason
                        }])

            Delivery_Error_Report = pd.concat([Delivery_Error_Report, new_line],ignore_index=True)

        error_reason = ''

In [ ]:
# Second, we get the street detail, that is normally the second group of the split

for index, row in df3.iterrows():
    
    customer_ID = row['ID']
    
    if customer_ID in Delivery_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Delivery_Address'].strip('_').strip('.').split('_')

        street_details = parts[-2] # geting the second group of the split

        last_index = find_last_number_index(street_details) # geting the index of the last number, which represents the street number

        street_no = street_details[:last_index+1].strip(',').strip('.') # street no is everything until the index of the last number

        street_name = street_details[last_index+1:] # street name starts right after the last number

        street_name = street_name.strip(',').strip('`').strip().strip('.')

        delivery_first_group.loc[delivery_first_group['ID'] == customer_ID, 'Delivery_StreetNo'] = street_no
        delivery_first_group.loc[delivery_first_group['ID'] == customer_ID, 'Delivery_StreetName'] = street_name

#### Billing Address 2

In [ ]:
# Third, we get what we have in the tird group for those who have something and put in the Billing Address 2 column
for index, row in df3.iterrows():
    
    customer_ID = row['ID']
    
    if customer_ID in Delivery_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Delivery_Address'].strip('_').strip('.').split('_')

        trading_as = []
        adress2 = []

        if len(parts) > 2:

            for part in parts[:-2]:
                if part.lower().startswith("t/a") or part.lower().startswith("t/as"):
                    trading_as.append(part.strip().strip(',').strip('.'))
                else:
                    adress2.append(part.strip().strip(',').strip('.'))


            if len(adress2) > 0:
                delivery_address2 = '\n'.join(adress2) # geting the second group of the split
                delivery_address2 = delivery_address2.strip('\n').strip('.')
                delivery_first_group.loc[delivery_first_group['ID'] == customer_ID, 'Delivery_Address2'] = delivery_address2

            if len(trading_as) > 0:
                delivery_ta = '\n'.join(trading_as)
                delivery_ta = delivery_ta.strip().strip(',')
                delivery_first_group.loc[delivery_first_group['ID'] == customer_ID, 'Delivery_Customisable_Value'] = delivery_ta
                delivery_ta = None

#### Updating the Delivery Address

In [ ]:
for index, row in df3.iterrows():
    
    customer_ID = row['ID']
    
    if customer_ID in Delivery_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Delivery_Address'].strip('_').strip('.').split('_') # spliting the string by _ that represents a line break

        suburb_details = parts[-1].split() # the last part is usually where the Suburb, post code and state are 

        post_code = suburb_details[-1].strip().strip(',').strip('.') # the post code is normally the last word in the last line

        state = suburb_details[-2].upper().strip().strip(',').strip('.') # the state is normally the second to last

        suburb = ' '.join(suburb_details[:-2]).upper().strip(',').strip('.') # suburb is the rest, it can be 2 or more words, so we join them with a space

        delivery_address = '\n'.join(parts[:-1])

        delivery_address += f'\n{suburb} {state} {post_code}'

        delivery_first_group.loc[delivery_first_group['ID'] == customer_ID, 'Delivery_Address'] = delivery_address

In [ ]:
delivery_first_group = delivery_first_group[~delivery_first_group['ID'].isin(Delivery_IDs_To_Exclude)]

delivery_first_group[['Customer_Code', 'ID', 'Delivery_Address', 'Delivery_Customisable_Value']]

### Second Group: PO Box and Locked Bag

In [ ]:
# Establish the connection with Windows authentication

#conn = pyodbc.connect(
#    f'DRIVER=ODBC Driver 17 for SQL Server;SERVER={server};DATABASE={database};UID={username};PWD={password};'
#)

# Create a cursor object to interact with the database

#cursor = conn.cursor()


select_delivery_second = f"""

WITH UpdateSelection AS (

SELECT DISTINCT [Source_Id]
FROM [sample_database].[dbo].[VA_DBT_Master]
WHERE [Audited_Field] IN ('Delivery_Address') 
AND Operation IN ('U', 'I')
AND Audit_DateTime > '{last_finish_date}'
AND Audit_User NOT LIKE 'ServiceAccount'

AND [After_Value] IS NOT NULL AND [After_Value] NOT LIKE 'TBA%' AND

([After_Value]  LIKE '%Locked Bag%' OR [After_Value] LIKE '%PO BOX%' OR [After_Value] LIKE '%P.O. BOX%' OR [After_Value] LIKE '%P.O BOX%'
 OR [After_Value] LIKE '%P O Box%' OR [After_Value] LIKE '%Private Bag%')

)

SELECT     
    [Customer_Code],
    [ID],
    REPLACE([Delivery_Address],CHAR(13)+CHAR(10),'_') AS [Delivery_Address]
FROM [sample_database].[dbo].[DBT_Master]

WHERE 

[Delivery_Address] IS NOT NULL AND [Delivery_Address] NOT LIKE 'TBA%' AND

([Delivery_Address]  LIKE '%Locked Bag%' OR [Delivery_Address] LIKE '%PO BOX%' OR [Delivery_Address] LIKE '%P.O. BOX%' OR [Delivery_Address] LIKE '%P.O BOX%'
 OR [Delivery_Address] LIKE '%P O Box%' OR [Delivery_Address] LIKE '%Private Bag%')

AND [ID] IN (SELECT DISTINCT [Source_Id]
            FROM UpdateSelection)
            
"""

#df4 = pd.read_sql_query(select_delivery_second, conn)

#cursor.close()
#conn.close()

df4 = pd.read_excel("Delivery Address - PO Boxes.xlsx")

df4

In [ ]:
delivery_second_group = pd.DataFrame(columns=columns)

delivery_second_group['Customer_Code'] = df4['Customer_Code']
delivery_second_group['ID'] = df4['ID']
delivery_second_group['Delivery_Address'] = df4['Delivery_Address']

delivery_second_group

#### Post Code, State and Suburb

In [ ]:
# testing if there's any row not in the pattern

error_reason = ''

for index, row in df4.iterrows():
    
    customer_ID = row['ID']
    
    parts = row['Delivery_Address'].strip('_').strip('.').split('_') # spliting the string by _ that represents a line break
    
    if len(parts) < 2:
        Delivery_IDs_To_Exclude.append(customer_ID)
        error_reason += 'Address is not complete. It should have at least 2 lines!\n'
        
        new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Billing_Address': row['Billing_Address'].replace("_", "\n"),
                                                    'Billing Error Type': error_reason
                        }])
        
        Delivery_Error_Report = pd.concat([Delivery_Error_Report, new_line],ignore_index=True)
        
        error_reason = ''
        
        continue
        
    else:
    
        suburb_details = parts[-1].split() # the last part is usually where the Suburb, post code and state are 
        
        if len(suburb_details) >= 2:
            state = suburb_details[-2].upper().strip(',').strip('`').strip('.') # the state is normally the second to last
        else:
            # Handle the case when the length is less than 2
            state = ''
            error_reason += 'Wrong Format. Please make sure the format looks like:\n{Address1}\n{Address2}\n{StreetNumber} {StreetName} {StreetType}\n{Suburb} {State} {Postcode}\n'
            Delivery_IDs_To_Exclude.append(customer_ID)
            
            new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Delivery_Address': row['Delivery_Address'].replace("_", "\n"),
                                                    'Delivery Error Type': error_reason
                        }])
            
            Delivery_Error_Report = pd.concat([Delivery_Error_Report, new_line],ignore_index=True)
        
            error_reason = ''
        
            continue
            
           #----------------------------------------------------------------------------------------------------------------------------------------------------------# 

        post_code = suburb_details[-1].strip(',').strip('`').strip().strip('.') # the post code is normally the last word in the last line

        suburb = ' '.join(suburb_details[:-2]).upper().strip(',').strip('.') # suburb is the rest, it can be 2 or more words, so we join them with a space

        # Check if the post_code looks like 4 numbers
        is_post_code_valid = re.match(r'^\d{4}$', post_code) is not None

        # Check if the state looks like 3 or 2 letters
        is_state_valid = re.match(r'^[A-Za-z]{2,3}$', state) is not None
        
        keywords_to_exclude = ['Locked Bag', 'PO BOX', 'P.O. BOX', 'P.O BOX', 'P O Box', 'Private Bag']
            
            
        if any(keyword.upper() in suburb for keyword in keywords_to_exclude):
            Delivery_IDs_To_Exclude.append(customer_ID)
            error_reason += 'Suburb contains address words.'


        
        if is_post_code_valid == False:
            if isinstance(post_code, int):
                error_reason += 'Postcode invalid. It should be 4 numbers.\n'
                Delivery_IDs_To_Exclude.append(customer_ID)
                
                
            if not any(re.match(r'^\d{4}$', str(item)) is not None for item in suburb_details):
                error_reason += 'Postcode Missing.\n'
                Delivery_IDs_To_Exclude.append(customer_ID)
                
                if not any(state in suburb_details for state in states):
                    error_reason += 'State Missing.\n'
                    
                    
                new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Delivery_Address': row['Delivery_Address'].replace("_", "\n"),
                                                    'Delivery Error Type': error_reason
                        }])
            
                Delivery_Error_Report = pd.concat([Delivery_Error_Report, new_line],ignore_index=True)

                error_reason = ''
                
                continue

                
            else:
                
                if not any(state in suburb_details for state in states):
                    error_reason += 'State Missing.\n'
                    Delivery_IDs_To_Exclude.append(customer_ID)
                    
                    new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Delivery_Address': row['Delivery_Address'].replace("_", "\n"),
                                                    'Delivery Error Type': error_reason
                        }])
            
                    Delivery_Error_Report = pd.concat([Delivery_Error_Report, new_line],ignore_index=True)

                    error_reason = ''
                    
                    continue
                    
                else:
                    error_reason += 'Wrong Format. Please make sure the format looks like:\n{Address1}\n{Address2}\n{StreetNumber} {StreetName} {StreetType}\n{Suburb} {State} {Postcode}\n'
                    Delivery_IDs_To_Exclude.append(customer_ID)
                    
                    new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Delivery_Address': row['Delivery_Address'].replace("_", "\n"),
                                                    'Delivery Error Type': error_reason
                        }])
            
                    Delivery_Error_Report = pd.concat([Delivery_Error_Report, new_line],ignore_index=True)

                    error_reason = ''
                    
                    continue
        
        
        
        if is_state_valid == False:
            
            if not any(state in suburb_details for state in states):
                error_reason += 'State Missing.\n'
                Delivery_IDs_To_Exclude.append(customer_ID)
            else:
                Delivery_IDs_To_Exclude.append(customer_ID)
                error_reason += 'State invalid. It should be 2 or 3 characters.\n'
            
        
        if suburb == '':
            error_reason += 'Suburb is Missing.'
            Delivery_IDs_To_Exclude.append(customer_ID)
        
            
        if is_post_code_valid is False or is_state_valid is False or suburb == '' or any(keyword.upper() in suburb for keyword in keywords_to_exclude):
            new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_ID,
                                                    'Delivery_Address': row['Delivery_Address'].replace("_", "\n"),
                                                    'Delivery Error Type': error_reason
                        }])
            
            Delivery_Error_Report = pd.concat([Delivery_Error_Report, new_line],ignore_index=True)
                
        error_reason = ''

In [ ]:
# Firt, we separate the Suburb, Post Code and State and put them into the new dataset

# The pattern we are looking for is: Condell Park NSW 2200 (Suburb State Post Code)

for index, row in df4.iterrows():
    
    customer_ID = row['ID']
    
    if customer_ID in Delivery_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Delivery_Address'].strip('_').strip('.').split('_') # spliting the string by _ that represents a line break

        suburb_details = parts[-1].split() # the last part is usually where the Suburb, post code and state are 

        post_code = suburb_details[-1].strip(',').strip('`').strip().strip('.') # the post code is normally the last word in the last line

        state = suburb_details[-2].upper().strip(',').strip('`').strip('.') # the state is normally the second to last

        suburb = ' '.join(suburb_details[:-2]).upper().strip(',').strip('.') # suburb is the rest, it can be 2 or more words, so we join them with a space

        delivery_second_group.loc[delivery_second_group['ID'] == customer_ID, 'Delivery_Postcode'] = post_code
        delivery_second_group.loc[delivery_second_group['ID'] == customer_ID, 'Delivery_State'] = state
        delivery_second_group.loc[delivery_second_group['ID'] == customer_ID, 'Delivery_Suburb'] = suburb

#### Billing Address 2

It was the billing address 2 the field that would receive the values, but after some problems in converting quotes in Salesforce, this field was changed to Billing_StreetName.

Reffer to this change in the TASK# 002370 - Split Billing Address - PO Box Cases (Baseplan)

In [ ]:
for index, row in df4.iterrows():
    
    customer_ID = row['ID']
    
    if customer_ID in Delivery_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Delivery_Address'].strip('_').strip('.').split('_') # spliting the string by _ that represents a line break

        trading_as = []
        adress2 = []
        streetname = []

        for part in parts[:-1]:
                if part.lower().startswith("t/a") or part.lower().startswith("t/as"):
                    trading_as.append(part.strip().strip(',').strip('.'))
                elif re.search(r'\bbox\b|\bbag\b', part, re.IGNORECASE):
                    streetname.append(part.strip().strip(',').strip('.'))
                else:                       
                    adress2.append(part.strip().strip(',').strip('.'))

        if len(adress2) > 0:
            delivery_address2 = '\n'.join(adress2) # geting the second group of the split
            delivery_address2 = delivery_address2.strip('\n').strip('.')
            delivery_second_group.loc[delivery_second_group['ID'] == customer_ID, 'Delivery_Address2'] = delivery_address2
            
        if len(streetname) > 0:
            delivery_streetName = '\n'.join(streetname) # geting the second group of the split
            delivery_streetName.strip('\n').strip().strip(',').strip('.')
            delivery_second_group.loc[delivery_second_group['ID'] == customer_ID, 'Delivery_StreetName'] = streetname

        if len(trading_as) > 0:
            delivery_ta = '\n'.join(trading_as)
            delivery_ta = delivery_ta.strip().strip(',')
            delivery_second_group.loc[delivery_second_group['ID'] == customer_ID, 'Delivery_Customisable_Value'] = delivery_ta
            delivery_ta = None

#### Updating the Delivery Address

In [ ]:
for index, row in df4.iterrows():
    
    customer_ID = row['ID']
    
    if customer_ID in Delivery_IDs_To_Exclude:
        continue
        
    else:
    
        parts = row['Delivery_Address'].strip('_').strip('.').split('_') # spliting the string by _ that represents a line break

        suburb_details = parts[-1].split() # the last part is usually where the Suburb, post code and state are 

        post_code = suburb_details[-1].strip().strip(',').strip('.') # the post code is normally the last word in the last line

        state = suburb_details[-2].upper().strip().strip(',').strip('.') # the state is normally the second to last

        suburb = ' '.join(suburb_details[:-2]).upper().strip(',').strip('.') # suburb is the rest, it can be 2 or more words, so we join them with a space

        delivery_address = '\n'.join(parts[:-1])

        delivery_address += f'\n{suburb} {state} {post_code}'

        delivery_second_group.loc[delivery_second_group['ID'] == customer_ID, 'Delivery_Address'] = delivery_address

In [ ]:
delivery_second_group = delivery_second_group[~delivery_second_group['ID'].isin(Delivery_IDs_To_Exclude)]

delivery_second_group

In [ ]:
values_before_df = pd.concat([df, df2, df3, df4], ignore_index=True)

values_before_df

# Uniting the dataframes

In [ ]:
# Establish the connection with Windows authentication

#conn = pyodbc.connect(
#    f'DRIVER=ODBC Driver 17 for SQL Server;SERVER={server};DATABASE={database};UID={username};PWD={password};'
#)

# Create a cursor object to interact with the database

#cursor = conn.cursor()

select_all_ids = """
SELECT     
	[Customer_Code],
    [ID]
FROM [sample_database].[dbo].[DBT_Master]
ORDER BY ID
"""

#Addresses_Complete = pd.read_sql_query(select_all_ids, conn)

#cursor.close()
#conn.close()

Addresses_Complete = pd.read_excel("All Customers.xlsx")

for col in columns[2:]:
    Addresses_Complete[col] = None


Addresses_Complete

In [ ]:
for index, row in Addresses_Complete.iterrows():
    
    customer_id = row['ID']
    
    # Find the corresponding row in the other DataFrames based on 'ID'
    billing_first_row = billing_first_group[billing_first_group['ID'] == customer_id]
    billing_second_row = billing_second_group[billing_second_group['ID'] == customer_id]
    delivery_first_row = delivery_first_group[delivery_first_group['ID'] == customer_id]
    delivery_second_row = delivery_second_group[delivery_second_group['ID'] == customer_id]
    
    # Check if there are matching rows in the DataFrames
    if not billing_first_row.empty:
        for column in columns[2:]:
            value = billing_first_row[column].values[0]
            if pd.isna(value):
                continue
            else:
                Addresses_Complete.at[index, column] = value

    if not billing_second_row.empty:
        for column in columns[2:]:
            value = billing_second_row[column].values[0]
            if pd.isna(value):
                continue
            else:
                Addresses_Complete.at[index, column] = value

    if not delivery_first_row.empty:
        for column in columns[2:]:
            value = delivery_first_row[column].values[0]
            if pd.isna(value):
                continue
            else:
                Addresses_Complete.at[index, column] = value

    if not delivery_second_row.empty:
        for column in columns[2:]:
            value = delivery_second_row[column].values[0]
            if pd.isna(value):
                continue
            else:
                Addresses_Complete.at[index, column] = value

In [ ]:
# Dropping the line where all values are null
Addresses_Complete = Addresses_Complete.dropna(subset = Addresses_Complete.columns[2:], how='all')

Addresses_Complete.reset_index(drop = True, inplace = True)

Addresses_Complete

# Updating the database

This part gets the list of all suburbs from the system.

In [ ]:
# Establish the connection with Windows authentication

#conn = pyodbc.connect(
#    f'DRIVER=ODBC Driver 17 for SQL Server;SERVER={server};DATABASE={database};UID={username};PWD={password};'
#)

# Create a cursor object to interact with the database
#cursor = conn.cursor()

select_suburb_details = """
SELECT SUB.[Id]
      ,[SuburbName]
	  ,STA.[StateName]
      ,[StateID]
      ,[Postcode]
  FROM [sample_database].[dbo].[ADR_Suburb] SUB
	LEFT JOIN [sample_database].[dbo].[ADR_State] STA
		ON SUB.[StateID] = STA.[Id]
"""

#suburbs = pd.read_sql_query(select_suburb_details, conn)

#cursor.close()
#conn.close()

suburbs = pd.read_excel("Suburbs List.xlsx")

suburbs

In [ ]:
# Select the columns for billing information and 'Customer_Code' and 'ID'
billing_columns = ['Customer_Code', 'ID', 'Billing_Address', 'Billing_StreetNo', 'Billing_StreetName', 'Billing_StreetType',
                   'Billing_Suburb', 'Billing_State', 'Billing_Postcode', 'Billing_Customisable_Value', 'Billing_Address2']

billing_df = Addresses_Complete[billing_columns]

# Select the columns for delivery information and 'Customer_Code' and 'ID'
delivery_columns = ['Customer_Code', 'ID', 'Delivery_Address', 'Delivery_StreetNo', 'Delivery_StreetName', 'Delivery_StreetType',
                    'Delivery_Suburb', 'Delivery_State', 'Delivery_Postcode', 'Delivery_Customisable_Value', 'Delivery_Address2']

delivery_df = Addresses_Complete[delivery_columns]

In [ ]:
billing_df = billing_df.dropna(subset=billing_df.columns[2:], how='all')

billing_df

## Inserting the addresses and retrieving the IDs

### Billing Addresses

This part checks if the suburb, postcode and state inserted by the user exists using the created functions explained before.

In [ ]:
# Testing to see if the suburb and postcode macthes

billing_insert_statments = ''

check_file = ''

Billing_IDs_To_Skip = []

error_reason = ''

for index, row in billing_df.iterrows():
    
    suburb = row['Billing_Suburb']
    original_suburb = row['Billing_Suburb']
    postcode = row['Billing_Postcode']
    original_postcode = row['Billing_Postcode']
    state = row['Billing_State']
    customer_id = row['ID']
    
    
    if suburb == 'HRMC':
        suburb = 'HUNTER REGION MC'
    elif suburb == 'GCMC' or suburb == 'GCMC BUNDALL':
        suburb = 'GOLD COAST MC'
    elif suburb == 'QVB':
        suburb = 'QUEEN VICTORIA BUILDING'
    
    filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb) & (suburbs['Postcode'] == postcode)]
    
    
    if filtered_suburbs.empty:
        
        filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb) & (suburbs['StateName'] == state)]
        
        if len(filtered_suburbs)>1:
            best_postcode_diff = float('inf')
            best_match = None
            
            for _, current_row in filtered_suburbs.iterrows():
                current_postcode = current_row['Postcode']
                postcode_diff = abs(int(current_postcode) - int(original_postcode))

                if postcode_diff < best_postcode_diff:
                    best_postcode_diff = postcode_diff
                    suburb = current_row['SuburbName']
                    postcode = current_row['Postcode']
            
            filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb) & (suburbs['Postcode'] == postcode)]
            check_file += f'\nCheck 1\n'
        
        elif filtered_suburbs.empty:
            matching_suburb = find_matching_suburb(suburb, postcode, suburbs)
        
            if matching_suburb:
                suburb = matching_suburb
                filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb.upper()) & (suburbs['Postcode'] == postcode)]
                check_file += f'\nCheck 2\n'

            else:
                suburb = original_suburb
                matching_split = find_matching_suburb_split(suburb, postcode, suburbs)

                if matching_split:
                    suburb = matching_split
                    filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb.upper()) & (suburbs['Postcode'] == postcode)]
                    check_file += f'\nCheck 3\n'

                else:
                    suburb = original_suburb
                    matching_split_state = find_matching_suburb_split_state(suburb, state, postcode, suburbs)

                    if matching_split_state[0]:
                        suburb = matching_split_state[0]
                        postcode = matching_split_state[1]

                        filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper()== suburb.upper()) & (suburbs['Postcode'] == postcode)]
                        check_file += f'\nCheck 4\n'
            
            
            
    if filtered_suburbs.empty:
        check_file += f'\n {index}---{customer_id} --- {original_suburb} --- !!!!!!!!!!!!!!!! EMPTY !!!!!!!!!!!!!!!!! \n'
        Billing_IDs_To_Skip.append(customer_id)
        error_reason += 'Postcode and Suburb does not match.\n'
        
        
    else:
        suburb_id = filtered_suburbs['Id'].values[0]
        state_id = filtered_suburbs['StateID'].values[0]
        postcode = filtered_suburbs['Postcode'].values[0]
        suburb = filtered_suburbs['SuburbName'].values[0]
        
        # Getting the new postcode and suburb to update the billing address
        
        parts = row['Billing_Address'].split('\n') # spliting the string by _ that represents a line break
    
        billing_address = '\n'.join(parts[:-1])

        billing_address += f'\n{suburb} {state} {postcode}'
        
        
        street_number = row['Billing_StreetNo']
        
        if street_number is not None:
            len_stnumber = len(street_number)
            street_number = street_number.replace("'", "''") 
            street_number = street_number.replace("\n", "'+char(13)+char(10)+'")
            street_number = f"'{street_number}'"
        else:
            street_number = 'NULL'
            len_stnumber = len(street_number)

        
        street_name = row['Billing_StreetName']
        if street_name is not None:
            len_stname = len(street_name)
            street_name = street_name.replace("'", "''")
            street_name = street_name.replace("\n", "'+char(13)+char(10)+'")
            street_name = f"'{street_name}'"
        else:
            street_name = 'NULL'
            len_stname = len(street_name)
        
        address = billing_address
        if address is not None:
            len_add = len(address)
            address = address.replace("'", "''")
            address = address.replace("\n", "'+char(13)+char(10)+'")
        else:
            address = 'NULL'
            len_add = len(address)

        address1 = row['Billing_Customisable_Value']
        if address1 is not None:
            len_add1 = len(address1)
            address1 = address1.replace("'", "''")
            address1 = address1.replace("\n", "'+char(13)+char(10)+'")
            address1 = f"'{address1}'"
        else:
            address1 = 'NULL'
            len_add1 = len(address1)
            

        address2 = row['Billing_Address2']
        if address2 is not None:
            len_add2 = len(address2)
            address2 = address2.replace("'", "''")
            address2 = address2.replace("\n", "'+char(13)+char(10)+'")
            address2 = f"'{address2}'"
        else:
            address2 = 'NULL'
            len_add2 = len(address2)

        
        
        if postcode != original_postcode or suburb != original_suburb:
            check_file += f'\n!!!DIFFERENT!!!\nOriginal Suburb: {original_suburb} --- Original Postcode: {original_postcode}\nIndex: {index} \nMatching Sub: {suburb}\nPostcode: {postcode}\nID: {customer_id} \n'
        
        else:
            check_file += f'\nOriginal Suburb: {original_suburb} --- Original Postcode: {original_postcode}\nIndex: {index} \nMatching Sub: {suburb}\nPostcode: {postcode}\nID: {customer_id} \n'
    
        insert = f"""
        INSERT INTO [sample_database].[dbo].[ADR_Address]
          ([Address]
          ,[Address1]
          ,[StreetNumber]
          ,[StreetName]
          ,[Address2]
          ,[SuburbID]
          ,[StateID]
          ,[CountryID]
          ,[Postcode])
         VALUES
               ('{address}' ---- {f'Greater {customer_id}' if len_add > 1000 else 'Smaller'}
               ,{address1} ---- {f'Greater {customer_id}' if len_add1 > 250 else 'Smaller'}
               ,{street_number} ---- {f'Greater {customer_id}' if len_stnumber > 20 else 'Smaller'}
               ,{street_name} ---- {f'Greater {customer_id}' if len_stname > 100 else 'Smaller'}
               ,{address2} ---- {f'Greater {customer_id}' if len_add2 > 70 else 'Smaller'}
               ,{suburb_id}
               ,{state_id}
               ,36
               ,{postcode})
        """
        
        billing_insert_statments += insert
        
        
        if len_add > 1000:
            Billing_IDs_To_Skip.append(customer_id)
            error_reason += 'Lenght of the Billing_Address is longer than 1000 characters.\n'
        
        if len_add1 > 250:
            Billing_IDs_To_Skip.append(customer_id)
            error_reason += 'Lenght of the Billing_Customisable_Value is longer than 250 characters.\n'
        
        if len_stnumber > 20:
            Billing_IDs_To_Skip.append(customer_id)
            error_reason += 'Lenght of the StreetNumber is longer than 20 characters.\n'
        
        if len_stname > 100:
            Billing_IDs_To_Skip.append(customer_id)
            error_reason += 'Lenght of the StreetName is longer than 100 characters.\n'
            
        if len_add2 > 70:
            Billing_IDs_To_Skip.append(customer_id)
            error_reason += 'Lenght of the Address2 is longer than 70 characters.\n'
            
            
        if any([len_add > 1000, len_add1 > 250, len_stnumber > 20, len_stname > 100, len_stname > 100, len_add2 > 70]):
            new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_id,
                                                    'Billing_Address': row['Billing_Address'].replace("_", "\n"),
                                                    'Billing Error Type': error_reason
                        }])
            
            Billing_Error_Report = pd.concat([Billing_Error_Report, new_line],ignore_index=True)
                
        error_reason = ''

In [ ]:
# Get the current date and time as a string
date_time = datetime.now().strftime("%Y%m%d_%H%M%S")

This part saves the values as they were before the modifications just in case we need to come back to them.

In [ ]:
folder_path = f'\\\choose\\your\\path{date_time}'

#file_path = os.path.join(folder_path, f'Values_Before.xlsx')

#values_before_df.to_excel(file_path, index=False)

After the checks, the code inserts into the addresses table the fixed customer addresses and gets the generated ID to use in the next step.

In [ ]:
# Establish the connection with Windows authentication

#conn = pyodbc.connect(
#    f'DRIVER=ODBC Driver 17 for SQL Server;SERVER={server};DATABASE={database};UID={username};PWD={password};'
#)

# Create a cursor object to interact with the database
#cursor = conn.cursor()

billing_insert_statments = ''

check_file = ''


for index, row in billing_df.iterrows():
    
    
    suburb = row['Billing_Suburb']
    original_suburb = row['Billing_Suburb']
    postcode = row['Billing_Postcode']
    original_postcode = row['Billing_Postcode']
    state = row['Billing_State']
    customer_id = row['ID']
    
    if customer_id in Billing_IDs_To_Skip:
        continue
    
    if suburb == 'HRMC':
        suburb = 'HUNTER REGION MC'
    elif suburb == 'GCMC' or suburb == 'GCMC BUNDALL':
        suburb = 'GOLD COAST MC'
    elif suburb == 'QVB':
        suburb = 'QUEEN VICTORIA BUILDING'
    
    filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb) & (suburbs['Postcode'] == postcode)]
    
    
    if filtered_suburbs.empty:
        
        filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb) & (suburbs['StateName'] == state)]
        
        if len(filtered_suburbs)>1:
            best_postcode_diff = float('inf')
            best_match = None
            
            for _, current_row in filtered_suburbs.iterrows():
                current_postcode = current_row['Postcode']
                postcode_diff = abs(int(current_postcode) - int(original_postcode))

                if postcode_diff < best_postcode_diff:
                    best_postcode_diff = postcode_diff
                    suburb = current_row['SuburbName']
                    postcode = current_row['Postcode']
            
            filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb) & (suburbs['Postcode'] == postcode)]
        
        elif filtered_suburbs.empty:
            matching_suburb = find_matching_suburb(suburb, postcode, suburbs)
        
            if matching_suburb:
                suburb = matching_suburb
                filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb.upper()) & (suburbs['Postcode'] == postcode)]

            else:
                suburb = original_suburb
                matching_split = find_matching_suburb_split(suburb, postcode, suburbs)

                if matching_split:
                    suburb = matching_split
                    filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb.upper()) & (suburbs['Postcode'] == postcode)]

                else:
                    suburb = original_suburb
                    matching_split_state = find_matching_suburb_split_state(suburb, state, postcode, suburbs)

                    if matching_split_state[0]:
                        suburb = matching_split_state[0]
                        postcode = matching_split_state[1]

                        filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper()== suburb.upper()) & (suburbs['Postcode'] == postcode)]
            
            
    if filtered_suburbs.empty:
        continue
        
        
    else:
        suburb_id = filtered_suburbs['Id'].values[0]
        state_id = filtered_suburbs['StateID'].values[0]
        postcode = filtered_suburbs['Postcode'].values[0]
        suburb = filtered_suburbs['SuburbName'].values[0]
        
        # Getting the new postcode and suburb to update the billing address
        
        parts = row['Billing_Address'].split('\n') # spliting the string by _ that represents a line break
    
        billing_address = '\n'.join(parts[:-1])

        billing_address += f'\n{suburb} {state} {postcode}'
        
        
        street_number = row['Billing_StreetNo']
        
        if street_number is not None:
            street_number = street_number.replace("'", "''") 
            street_number = street_number.replace("\n", "'+char(13)+char(10)+'")
            street_number = f"'{street_number}'"
        else:
            street_number = 'NULL'
            len_stnumber = len(street_number)

        
        street_name = row['Billing_StreetName']
        if street_name is not None:
            street_name = street_name.replace("'", "''") 
            street_name = street_name.replace("\n", "'+char(13)+char(10)+'")
            street_name = f"'{street_name}'"
        else:
            street_name = 'NULL'
            len_stname = len(street_name)
        
        address = billing_address
        if address is not None:
            address = address.replace("'", "''")
            address = address.replace("\n", "'+char(13)+char(10)+'")
        else:
            address = 'NULL'

        address1 = row['Billing_Customisable_Value']
        if address1 is not None:
            address1 = address1.replace("'", "''")
            address1 = address1.replace("\n", "'+char(13)+char(10)+'")
            address1 = f"'{address1}'"
        else:
            address1 = 'NULL'
            len_add1 = len(address1)
            

        address2 = row['Billing_Address2']
        if address2 is not None:
            address2 = address2.replace("'", "''")
            address2 = address2.replace("\n", "'+char(13)+char(10)+'")
            address2 = f"'{address2}'"
        else:
            address2 = 'NULL'
            len_add2 = len(address2)

        
       
        insert = f"""
        INSERT INTO [sample_database].[dbo].[ADR_Address]
          ([Address]
          ,[Address1]
          ,[StreetNumber]
          ,[StreetName]
          ,[Address2]
          ,[SuburbID]
          ,[StateID]
          ,[CountryID]
          ,[Postcode])
         VALUES
               ('{address}'
               ,{address1} 
               ,{street_number} 
               ,{street_name}
               ,{address2}
               ,{suburb_id}
               ,{state_id}
               ,36
               ,{postcode})
        """
        
        billing_insert_statments += insert
        
        # Execute each SQL query in the list
        #cursor.execute(insert)

        #conn.commit()
        
        
        select_id = """
SELECT MAX([ID])
FROM [sample_database].[dbo].[ADR_Address]
        """
        # Execute the select_id query
        #cursor.execute(select_id)
        
        # Fetch the result and extract the ID
        #new_id = cursor.fetchone()[0]
        
        #Addresses_Complete.loc[Addresses_Complete['ID'] == customer_id, 'Billing_ADR_Address_ID'] = new_id
        Addresses_Complete.loc[Addresses_Complete['ID'] == customer_id, 'Billing_Suburb'] = suburb
        Addresses_Complete.loc[Addresses_Complete['ID'] == customer_id, 'Billing_Postcode'] = postcode
        Addresses_Complete.loc[Addresses_Complete['ID'] == customer_id, 'Billing_Address'] = billing_address
       
    
# print(new_id)

# Close the cursor and connection
#cursor.close()
#conn.close()

### Delivery Addresses

In [ ]:
delivery_df = delivery_df.dropna(subset=delivery_df.columns[2:], how='all')

delivery_df

In [ ]:
delivery_insert_statments = ''

delivery_check_file = ''

Delivery_IDs_To_Skip = []

error_reason = ''

for index, row in delivery_df.iterrows():
    
    suburb = row['Delivery_Suburb']
    original_suburb = row['Delivery_Suburb']
    postcode = row['Delivery_Postcode']
    original_postcode = row['Delivery_Postcode']
    state = row['Delivery_State']
    customer_id = row['ID']
    
    if suburb == 'HRMC':
        suburb = 'HUNTER REGION MC'
    elif suburb == 'GCMC' or suburb == 'GCMC BUNDALL':
        suburb = 'GOLD COAST MC'
    elif suburb == 'QVB':
        suburb = 'QUEEN VICTORIA BUILDING'
        
    
    filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb) & (suburbs['Postcode'] == postcode)]
    
    
    if filtered_suburbs.empty:
        
        filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb) & (suburbs['StateName'] == state)]
        
        if len(filtered_suburbs)>1:
            best_postcode_diff = float('inf')
            best_match = None
            
            for _, current_row in filtered_suburbs.iterrows():
                current_postcode = current_row['Postcode']
                postcode_diff = abs(int(current_postcode) - int(original_postcode))

                if postcode_diff < best_postcode_diff:
                    best_postcode_diff = postcode_diff
                    suburb = current_row['SuburbName']
                    postcode = current_row['Postcode']
            
            filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb) & (suburbs['Postcode'] == postcode)]
            delivery_check_file += f'\nCheck 1\n'
        
        elif filtered_suburbs.empty:
            matching_suburb = find_matching_suburb(suburb, postcode, suburbs)
        
            if matching_suburb:
                suburb = matching_suburb
                filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb.upper()) & (suburbs['Postcode'] == postcode)]
                delivery_check_file += f'\nCheck 2\n'

            else:
                suburb = original_suburb
                matching_split = find_matching_suburb_split(suburb, postcode, suburbs)

                if matching_split:
                    suburb = matching_split
                    filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb.upper()) & (suburbs['Postcode'] == postcode)]
                    delivery_check_file += f'\nCheck 3\n'

                else:
                    suburb = original_suburb
                    matching_split_state = find_matching_suburb_split_state(suburb, state, postcode, suburbs)

                    if matching_split_state[0]:
                        suburb = matching_split_state[0]
                        postcode = matching_split_state[1]

                        filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper()== suburb.upper()) & (suburbs['Postcode'] == postcode)]
                        delivery_check_file += f'\nCheck 4\n'
            
            
            
    if filtered_suburbs.empty:
        delivery_check_file += f'\n {index}---{customer_id} --- {original_suburb} --- !!!!!!!!!!!!!!!! EMPTY !!!!!!!!!!!!!!!!! \n'
        Delivery_IDs_To_Skip.append(customer_id)
        error_reason += 'Postcode and Suburb does not match.\n'
        
        
    else:
        suburb_id = filtered_suburbs['Id'].values[0]
        state_id = filtered_suburbs['StateID'].values[0]
        postcode = filtered_suburbs['Postcode'].values[0]
        suburb = filtered_suburbs['SuburbName'].values[0]
        
        # Getting the new postcode and suburb to update the billing address
        
        parts = row['Delivery_Address'].split('\n') # spliting the string by _ that represents a line break
    
        delivery_address = '\n'.join(parts[:-1])

        delivery_address += f'\n{suburb} {state} {postcode}'
        
        
        street_number = row['Delivery_StreetNo']
        
        if street_number is not None:
            len_stnumber = len(street_number)
            street_number = street_number.replace("'", "''") 
            street_number = street_number.replace("\n", "'+char(13)+char(10)+'")
            street_number = f"'{street_number}'"
        else:
            street_number = 'NULL'
            len_stnumber = len(street_number)
        
        street_name = row['Delivery_StreetName']
        if street_name is not None:
            len_stname = len(street_name)
            street_name = street_name.replace("'", "''") 
            street_name = street_name.replace("\n", "'+char(13)+char(10)+'")
            street_name = f"'{street_name}'"
        else:
            street_name = 'NULL'
            len_stname = len(street_name)
        
        address = delivery_address
        if address is not None:
            len_add = len(address)
            address = address.replace("'", "''")
            address = address.replace("\n", "'+char(13)+char(10)+'")
        else:
            address = 'NULL'
            len_add = len(address)

        address1 = row['Delivery_Customisable_Value']
        if address1 is not None:
            len_add1 = len(address1)
            address1 = address1.replace("'", "''")
            address1 = address1.replace("\n", "'+char(13)+char(10)+'")
            address1 = f"'{address1}'"
        else:
            address1 = 'NULL'
            len_add1 = len(address1)

        address2 = row['Delivery_Address2']
        if address2 is not None:
            len_add2 = len(address2)
            address2 = address2.replace("'", "''")
            address2 = address2.replace("\n", "'+char(13)+char(10)+'")
            address2 = f"'{address2}'"
        else:
            address2 = 'NULL'
            len_add2 = len(address2)

        
        
        if postcode != original_postcode or suburb != original_suburb:
            delivery_check_file += f'\n!!!DIFFERENT!!!\nOriginal Suburb: {original_suburb} --- Original Postcode: {original_postcode}\nIndex: {index} \nMatching Sub: {suburb}\nPostcode: {postcode}\nID: {customer_id} \n'
        
        else:
            delivery_check_file += f'\nOriginal Suburb: {original_suburb} --- Original Postcode: {original_postcode}\nIndex: {index} \nMatching Sub: {suburb}\nPostcode: {postcode}\nID: {customer_id} \n'
    
        insert = f"""
        INSERT INTO [sample_database].[dbo].[ADR_Address]
          ([Address]
          ,[Address1]
          ,[StreetNumber]
          ,[StreetName]
          ,[Address2]
          ,[SuburbID]
          ,[StateID]
          ,[CountryID]
          ,[Postcode])
         VALUES
               ('{address}' ---- {f'Greater {customer_id}' if len(address) > 1000 else 'Smaller'}
               ,{address1} ---- {f'Greater {customer_id}' if len_add1 > 250 else 'Smaller'}
               ,{street_number} ---- {f'Greater {customer_id}' if len_stnumber > 20 else 'Smaller'}
               ,{street_name} ---- {f'Greater {customer_id}' if len_stname > 100 else 'Smaller'}
               ,{address2} ---- {f'Greater {customer_id}' if len_add2 > 70 else 'Smaller'}
               ,{suburb_id}
               ,{state_id}
               ,36
               ,{postcode})
        """
        
        delivery_insert_statments += insert
             
        
        if len_add > 1000:
            Delivery_IDs_To_Skip.append(customer_id)
            error_reason += 'Lenght of the Billing_Address is longer than 1000 characters.\n'
        
        if len_add1 > 250:
            Delivery_IDs_To_Skip.append(customer_id)
            error_reason += 'Lenght of the Billing_Customisable_Value is longer than 250 characters.\n'
        
        if len_stnumber > 20:
            Delivery_IDs_To_Skip.append(customer_id)
            error_reason += 'Lenght of the StreetNumber is longer than 20 characters.\n'
        
        if len_stname > 100:
            Delivery_IDs_To_Skip.append(customer_id)
            error_reason += 'Lenght of the StreetName is longer than 100 characters.\n'
            
        if len_add2 > 70:
            Delivery_IDs_To_Skip.append(customer_id)
            error_reason += 'Lenght of the Address2 is longer than 70 characters.\n'
            
            
        if any([len_add > 1000, len_add1 > 250, len_stnumber > 20, len_stname > 100, len_stname > 100, len_add2 > 70]):
            new_line = pd.DataFrame.from_records([{'Customer_Code': row['Customer_Code'],
                                                    'ID': customer_id,
                                                    'Delivery_Address': row['Delivery_Address'].replace("_", "\n"),
                                                    'Delivery Error Type': error_reason
                        }])
            
            Delivery_Error_Report = pd.concat([Delivery_Error_Report, new_line],ignore_index=True)
                
        error_reason = ''

This part was commented because it only works when a file is saved.

It saves the Error Report in an Excel file so it can be checked after what happened in each customer.

In [ ]:
folder_path = f'\\\choose\\your\\path\\{date_time}'

'''
file_path = os.path.join(folder_path, f'Error_Report.xlsx')

Error_Report = pd.merge(Billing_Error_Report, Delivery_Error_Report, on=['Customer_Code', 'ID'], how='outer')

error_report_columns = ['Customer_Code', 'ID'] + [col for col in Error_Report.columns if col not in ['Customer_Code', 'ID']]
Error_Report = Error_Report[error_report_columns]

Error_Report.to_excel(file_path, index=False)

# Create ExcelWriter and write DataFrame to Excel
with pd.ExcelWriter(file_path, engine='xlsxwriter') as writer:
    Error_Report.to_excel(writer, index=False, sheet_name='Sheet1')

    # Access the xlsxwriter workbook and worksheet objects from the ExcelWriter object.
    workbook = writer.book
    worksheet = writer.sheets['Sheet1']

    # Add a text wrap format.
    text_wrap_format = workbook.add_format({'text_wrap': True, 'align': 'center'})
    
    # Set the width of column 'A' to 26
    worksheet.set_column('A:A', 26)
    
    # Set the width of column 'B' to 10
    worksheet.set_column('B:B', 10)
    
    # Set the width of columns 'C' to 'F' to 55
    worksheet.set_column('C:F', 55, text_wrap_format)

'''

The next part will open the Error Report dataset and check if there's any line.

Case there are error lines, the code sends a request to a link an activate a PowerAutomate flow that get the error report file and sends via email to the responsible users.

In [ ]:
# Check if the Error_Report DataFrame has any lines

'''
if not Error_Report.empty:
    # The DataFrame has lines, proceed with further actions
    folder_path = f'\\\choose\\your\\path\\Error Report'
    
    #file_path = os.path.join(folder_path, f'Error_Report.xlsx')

    error_report_columns = ['Customer_Code', 'ID'] + [col for col in Error_Report.columns if col not in ['Customer_Code', 'ID']]
    Error_Report = Error_Report[error_report_columns]

    Error_Report.to_excel(file_path, index=False)

    # Create ExcelWriter and write DataFrame to Excel
    with pd.ExcelWriter(file_path, engine='xlsxwriter') as writer:
        Error_Report.to_excel(writer, index=False, sheet_name='Sheet1')

        # Access the xlsxwriter workbook and worksheet objects from the ExcelWriter object.
        workbook = writer.book
        worksheet = writer.sheets['Sheet1']

        # Add a text wrap format.
        text_wrap_format = workbook.add_format({'text_wrap': True, 'align': 'center'})
        
        # Set the width of column 'A' to 26
        worksheet.set_column('A:A', 26)
        
        # Set the width of column 'B' to 10
        worksheet.set_column('B:B', 10)
        
        # Set the width of columns 'C' to 'F' to 55
        worksheet.set_column('C:F', 55, text_wrap_format)
        
    time.sleep(5)
        
    url = 'write here the power automate flow link to activate the flow'

    # Send POST request to Power Automate flow
    requests.post(url)
    
'''

Now it inserts the delivery addresses values into the addresses table and gets the generated id to use in the next step. 

In [ ]:
# Establish the connection with Windows authentication

#conn = pyodbc.connect(
#    f'DRIVER=ODBC Driver 17 for SQL Server;SERVER={server};DATABASE={database};UID={username};PWD={password};'
#)

# Create a cursor object to interact with the database
#cursor = conn.cursor()

delivery_insert_statments = ''

delivery_check_file = ''


for index, row in delivery_df.iterrows():
    
    
    suburb = row['Delivery_Suburb']
    original_suburb = row['Delivery_Suburb']
    postcode = row['Delivery_Postcode']
    original_postcode = row['Delivery_Postcode']
    state = row['Delivery_State']
    customer_id = row['ID']
    
    if customer_id in Delivery_IDs_To_Skip:
        continue
    
    if suburb == 'HRMC':
        suburb = 'HUNTER REGION MC'
    elif suburb == 'GCMC' or suburb == 'GCMC BUNDALL':
        suburb = 'GOLD COAST MC'
    elif suburb == 'QVB':
        suburb = 'QUEEN VICTORIA BUILDING'
    
    filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb) & (suburbs['Postcode'] == postcode)]
    
    
    if filtered_suburbs.empty:
        
        filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb) & (suburbs['StateName'] == state)]
        
        if len(filtered_suburbs)>1:
            best_postcode_diff = float('inf')
            best_match = None
            
            for _, current_row in filtered_suburbs.iterrows():
                current_postcode = current_row['Postcode']
                postcode_diff = abs(int(current_postcode) - int(original_postcode))

                if postcode_diff < best_postcode_diff:
                    best_postcode_diff = postcode_diff
                    suburb = current_row['SuburbName']
                    postcode = current_row['Postcode']
            
            filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb) & (suburbs['Postcode'] == postcode)]
        
        elif filtered_suburbs.empty:
            matching_suburb = find_matching_suburb(suburb, postcode, suburbs)
        
            if matching_suburb:
                suburb = matching_suburb
                filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb.upper()) & (suburbs['Postcode'] == postcode)]

            else:
                suburb = original_suburb
                matching_split = find_matching_suburb_split(suburb, postcode, suburbs)

                if matching_split:
                    suburb = matching_split
                    filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper() == suburb.upper()) & (suburbs['Postcode'] == postcode)]

                else:
                    suburb = original_suburb
                    matching_split_state = find_matching_suburb_split_state(suburb, state, postcode, suburbs)

                    if matching_split_state[0]:
                        suburb = matching_split_state[0]
                        postcode = matching_split_state[1]

                        filtered_suburbs = suburbs[(suburbs['SuburbName'].str.upper()== suburb.upper()) & (suburbs['Postcode'] == postcode)]
            
            
            
    if filtered_suburbs.empty:
        continue
        
    else:
        suburb_id = filtered_suburbs['Id'].values[0]
        state_id = filtered_suburbs['StateID'].values[0]
        postcode = filtered_suburbs['Postcode'].values[0]
        suburb = filtered_suburbs['SuburbName'].values[0]
        
        # Getting the new postcode and suburb to update the billing address
        
        parts = row['Delivery_Address'].split('\n') # spliting the string by _ that represents a line break
    
        delivery_address = '\n'.join(parts[:-1])

        delivery_address += f'\n{suburb} {state} {postcode}'
        
        
        street_number = row['Delivery_StreetNo']
        
        if street_number is not None:
            len_stnumber = len(street_number)
            street_number = street_number.replace("'", "''") 
            street_number = street_number.replace("\n", "'+char(13)+char(10)+'")
            street_number = f"'{street_number}'"
        else:
            street_number = 'NULL'
            len_stnumber = len(street_number)
        
        street_name = row['Delivery_StreetName']
        if street_name is not None:
            len_stname = len(street_name)
            street_name = street_name.replace("'", "''") 
            street_name = street_name.replace("\n", "'+char(13)+char(10)+'")
            street_name = f"'{street_name}'"
        else:
            street_name = 'NULL'
            len_stname = len(street_name)
        
        address = delivery_address
        if address is not None:
            address = address.replace("'", "''")
            address = address.replace("\n", "'+char(13)+char(10)+'")
        else:
            address = 'NULL'

        address1 = row['Delivery_Customisable_Value']
        if address1 is not None:
            len_add1 = len(address1)
            address1 = address1.replace("'", "''")
            address1 = address1.replace("\n", "'+char(13)+char(10)+'")
            address1 = f"'{address1}'"
        else:
            address1 = 'NULL'
            len_add1 = len(address1)

        address2 = row['Delivery_Address2']
        if address2 is not None:
            len_add2 = len(address2)
            address2 = address2.replace("'", "''")
            address2 = address2.replace("\n", "'+char(13)+char(10)+'")
            address2 = f"'{address2}'"
        else:
            address2 = 'NULL'
            len_add2 = len(address2)

        
        
        
        insert = f"""
        INSERT INTO [sample_database].[dbo].[ADR_Address]
          ([Address]
          ,[Address1]
          ,[StreetNumber]
          ,[StreetName]
          ,[Address2]
          ,[SuburbID]
          ,[StateID]
          ,[CountryID]
          ,[Postcode])
         VALUES
               ('{address}'
               ,{address1}
               ,{street_number}
               ,{street_name}
               ,{address2}
               ,{suburb_id}
               ,{state_id}
               ,36
               ,{postcode})
        """
        
        delivery_insert_statments += insert
        
        # Execute each SQL query in the list
        
        #cursor.execute(insert)

        #conn.commit()
        
        
        select_id = """
SELECT MAX([ID])
FROM [sample_database].[dbo].[ADR_Address]
        """
        # Execute the select_id query
        #cursor.execute(select_id)
        
        # Fetch the result and extract the ID
        #new_id = cursor.fetchone()[0]
        
        #Addresses_Complete.loc[Addresses_Complete['ID'] == customer_id, 'Delivery_ADR_Address_ID'] = new_id
        Addresses_Complete.loc[Addresses_Complete['ID'] == customer_id, 'Delivery_Suburb'] = suburb
        Addresses_Complete.loc[Addresses_Complete['ID'] == customer_id, 'Delivery_Postcode'] = postcode
        Addresses_Complete.loc[Addresses_Complete['ID'] == customer_id, 'Delivery_Address'] = delivery_address
        
#print(new_id)

# Close the cursor and connection

#cursor.close()
#conn.close()

In [ ]:
# Set billing columns to None if ID is in Billing_IDs_To_Skip
Addresses_Complete.loc[Addresses_Complete['ID'].isin(Billing_IDs_To_Skip), [
    'Billing_ADR_Address_ID', 'Billing_Address', 'Billing_StreetNo',
    'Billing_StreetName', 'Billing_StreetType', 'Billing_Suburb',
    'Billing_State', 'Billing_Postcode', 'Billing_Customisable_Value',
    'Billing_Address2'
]] = None

# Set delivery columns to None if ID is in Delivery_IDs_To_Skip
Addresses_Complete.loc[Addresses_Complete['ID'].isin(Delivery_IDs_To_Skip), [
    'Delivery_ADR_Address_ID', 'Delivery_Address', 'Delivery_StreetNo',
    'Delivery_StreetName', 'Delivery_StreetType', 'Delivery_Suburb',
    'Delivery_State', 'Delivery_Postcode', 'Delivery_Customisable_Value',
    'Delivery_Address2'
]] = None


Addresses_Complete = Addresses_Complete.dropna(subset = Addresses_Complete.columns[2:], how='all')

Addresses_Complete.reset_index(drop = True, inplace = True)

Next steo saves the complete addresses table just for consultation if needed.

In [ ]:
folder_path = f'\\\choose\\your\\path\\{date_time}'

#file_path = os.path.join(folder_path, f'Addresses_Complete_Auto.xlsx')

#Addresses_Complete.to_excel(file_path, index=False)

## Updating the Master table

### Billing Fields

In [ ]:
billing_columns2 = ['Customer_Code', 'ID', 'Billing_ADR_Address_ID', 'Billing_Address', 'Billing_StreetNo', 'Billing_StreetName', 'Billing_StreetType',
                   'Billing_Suburb', 'Billing_State', 'Billing_Postcode', 'Billing_Customisable_Value', 'Billing_Address2']

# Select the columns for delivery information and 'Customer_Code' and 'ID'
delivery_columns2 = ['Customer_Code', 'ID', 'Delivery_ADR_Address_ID', 'Delivery_Address', 'Delivery_StreetNo', 'Delivery_StreetName', 'Delivery_StreetType',
                    'Delivery_Suburb', 'Delivery_State', 'Delivery_Postcode', 'Delivery_Customisable_Value', 'Delivery_Address2']

In [ ]:
Addresses_Complete_Billing = Addresses_Complete[billing_columns2]

Addresses_Complete_Billing = Addresses_Complete_Billing.dropna(subset = Addresses_Complete_Billing.columns[2:], how='all')

Addresses_Complete_Billing.reset_index(drop = True, inplace = True)

In [ ]:
Addresses_Complete_Billing

After all changes and fixes, the code updates the customer table in SQL to reflect the correct values.

In [ ]:
'''
# Establish the connection with Windows authentication
conn = pyodbc.connect(
    f'DRIVER=ODBC Driver 17 for SQL Server;SERVER={server};DATABASE={database};UID={username};PWD={password};'
)

# Create a cursor object to interact with the database
cursor = conn.cursor()

all_statments = ''


for index, row in Addresses_Complete_Billing.iterrows():
    
    customer_id = row['ID']

    if customer_id in (Billing_IDs_To_Skip + Billing_IDs_To_Exclude):
        continue
    
    update_string = ''
    count = 0
    
    for column in billing_columns2[2:]:
        
        value = Addresses_Complete_Billing.at[index, column]

        if value is not None:
            value = str(value)
            value = value.replace("'", "''")  # Escape single quotes
            value = value.replace("\n", "'+char(13)+char(10)+'")
    
            
            if count == 0:
                update_string += f"[{column}] = '{value}'\n"
                count += 1
            else:
                update_string += f",[{column}] = '{value}'\n"
                count += 1
    
    update_beginning = """UPDATE [sample_database].[dbo].[DBT_Master]
SET
 """
    
    update_end = f""",[Billing_Country] = 'Australia'
    WHERE [ID] = {customer_id};
    """
    
    
    sql_query = update_beginning + update_string + update_end
    
    all_statments += sql_query
    
    # Execute each SQL query in the list
    cursor.execute(sql_query)

    conn.commit()
    
# Close the cursor and connection
cursor.close()
conn.close()
'''

### Delivery Fields

In [ ]:
Addresses_Complete_Delivery = Addresses_Complete[delivery_columns2]

Addresses_Complete_Delivery = Addresses_Complete_Delivery.dropna(subset = Addresses_Complete_Delivery.columns[2:], how='all')

Addresses_Complete_Delivery.reset_index(drop = True, inplace = True)

In [ ]:
'''
# Establish the connection with Windows authentication
conn = pyodbc.connect(
    f'DRIVER=ODBC Driver 17 for SQL Server;SERVER={server};DATABASE={database};UID={username};PWD={password};'
)

# Create a cursor object to interact with the database
cursor = conn.cursor()


for index, row in Addresses_Complete_Delivery.iterrows():
    
    customer_id = row['ID']

    if customer_id in (Delivery_IDs_To_Skip + Delivery_IDs_To_Exclude):
        continue
    
    update_string = ''
    count = 0
    
    for column in delivery_columns2[2:]:
        
        value = Addresses_Complete_Delivery.at[index, column]

        if value is not None:
            value = str(value)
            value = value.replace("'", "''")  # Escape single quotes
            value = value.replace("\n", "'+char(13)+char(10)+'")
    
            
            if count == 0:
                update_string += f"[{column}] = '{value}'\n"
                count += 1
            else:
                update_string += f",[{column}] = '{value}'\n"
                count += 1
    
    update_beginning = """UPDATE [sample_database].[dbo].[DBT_Master]
SET
 """
    
    update_end = f""",[Delivery_Country] = 'Australia'
    WHERE [ID] = {customer_id};
    """
    
    
    sql_query = update_beginning + update_string + update_end
    
    all_statments += sql_query
    
    # Execute each SQL query in the list
    cursor.execute(sql_query)

    conn.commit()
    
# Close the cursor and connection
cursor.close()
conn.close()
'''

In [ ]:
# Set the finish date and time to the current moment
finish_date = datetime.now()

# Calculate the time spent
time_spent = finish_date - start_date

# Format dates and times as strings in the desired format
start_date_str = start_date.strftime('%d/%m/%y %H:%M:%S')
finish_date_str = finish_date.strftime('%d/%m/%y %H:%M:%S')

Lastly, it inserts into the log table the information of this run.

In [ ]:
'''
# Establish the connection with Windows authentication
conn = pyodbc.connect(
    f'DRIVER=ODBC Driver 17 for SQL Server;SERVER={server};DATABASE={database};UID={username};PWD={password};'
)

# Create a cursor object to interact with the database
cursor = conn.cursor()

start_date_SQL = start_date.strftime('%Y-%m-%d %H:%M:%S')
finish_date_SQL = finish_date.strftime('%Y-%m-%d %H:%M:%S')

if last_modified_date:
    last_modified = last_modified_date
    
else:
    last_modified = last_finish_date

no_rows = Addresses_Complete.shape[0]

insert_log = f"""

INSERT INTO [sample_database].[dbo].[TBL_DBT_Address_Updates_Log] (StartDate, FinishDate, TotalTime, No_Rows, Last_Modified_Date)
VALUES ('{start_date_SQL}', '{finish_date_SQL}', '{time_spent}', {no_rows}, '{last_modified}');

"""

cursor.execute(insert_log)

conn.commit()

# Close the cursor and connection
cursor.close()
conn.close()
'''